# HingRoBERTa & XLM-RoBERTa — Training + Cross-Lingual Testing + XAI Outputs

## Pipeline
| Step | Detail |
|---|---|
| **Normalize** | Script-aware normalization applied per dataset before training/testing |
| **Train** | `FINAL_NLP_DATASET.csv` — Hinglish (33,984 rows) → **Hinglish pipeline** |
| **Test 1** | `cleaned_final_english.csv` — English (24,783 rows) → **English pipeline** |
| **Test 2** | `cleaned_hindi_final.csv` — Hindi (8,192 rows) → **Devanagari pipeline** |
| **Models** | `l3cube-pune/hinglish-roberta` (HingRoBERTa) + `xlm-roberta-base` (XLM-RoBERTa) |
| **XAI** | Saves attentions, hidden states, logits, token rankings → SHAP / Attention / AOPC |

**Runtime:** GPU → Runtime > Change runtime type > T4 GPU

---
### Label Convention (confirmed from data inspection)
- `0` = Non-Hate / Clean
- `1` = Hate / Offensive

All three datasets use this same convention.

### Normalization strategy
Each dataset is normalized using its own source-language pipeline **before** tokenization:
- **Hinglish** (train): lowercase → emoji strip → URL/mention removal → Devanagari abuse romanisation → censored-word regex → slang canonicalisation → fuzzy profanity matching
- **English** (test): ftfy unicode fix → emoji demojize → lowercase → contraction expansion → URL/mention strip → elongation collapse
- **Devanagari/Hindi** (test): ftfy → emoji demojize → URL/mention strip → indicnlp DevanagariNormalizer (nukta, chandrabindu, anusvara, matras)


## Cell 1 — Install Dependencies

In [ ]:
%%capture
!pip install transformers==4.40.0 datasets scikit-learn accelerate emoji
!pip install python-Levenshtein joblib ftfy better_profanity indic-nlp-library


## Cell 2b — Normalization Pipelines
Script-aware normalization: **Hinglish** (romanised) → **English** → **Devanagari**.
The `normalize(text)` function auto-detects script and routes to the right pipeline.
Applied to training data (Hinglish) and both test sets before any model sees the text.

In [5]:
# =============================================================
# NORMALIZATION PIPELINES  (Hinglish | English | Devanagari)
# =============================================================
import re, unicodedata, warnings
from typing import List, Optional, Tuple
warnings.filterwarnings('ignore')

try:
    import emoji as _emoji_norm
    HAS_EMOJI = True
except ImportError:
    HAS_EMOJI = False

try:
    import ftfy
    HAS_FTFY = True
except ImportError:
    HAS_FTFY = False

try:
    from Levenshtein import distance as lev_distance
    HAS_LEV = True
except ImportError:
    HAS_LEV = False

try:
    from indicnlp.normalize.indic_normalize import DevanagariNormalizer
    _deva_normalizer = DevanagariNormalizer()
    HAS_INDIC = True
except ImportError:
    HAS_INDIC = False
    print('[WARN] indic-nlp-library not found — Devanagari uses regex fallback.')

# ── Profanity lists ────────────────────────────────────────────────────────
EN_PROFANITY_LIST = [
    "fuck", "shit", "ass", "bitch", "bastard", "damn", "cunt",
    "dick", "cock", "pussy", "whore", "slut", "piss", "crap",
    "asshole", "motherfucker", "fucker", "bullshit", "jackass",
    "dumbass", "dipshit", "shithead", "fuckhead", "goddamn",
]
HI_PROFANITY_LIST = [
    "madarchod","behenchod","bhenchod","maa ki aankh","mc","bc","bsdk",
    "chutiya","chut","chod","chodu","chinal","randi","raand","randwa","randwe",
    "gandu","gaand","gandfat","gandmara","bhosda","bhosdike","bhosdiwala","bhosdiwale",
    "lavda","loda","lode","lund","lulli","lauda","laude","laura","tatte","tatti",
    "jhaat","jhatu","harami","haraamkhor","haramzada","kamina","kamine","kameena",
    "kutta","kutte","kuttiya","suar","ullu","gadha","gadhe","bevakoof","bewakoof",
    "fattu","charsi","dalal","dalaal","dalla","bhadwa","bhadva","hijra","chakka",
    "ling","gaandmar","gandmasti","mut","moot","muth",
    "gaand mar","gaand mara","gaand marwa","bhosdi ke","bhosdike",
    "madarchod ke","behenchod ke","teri maa ki","teri behen ki",
    "maa chudaye","behen chudaye","saala","saali","bakchod","bakchodi",
    "jhant","jhantu","ullu ka pattha","kamchor","launda","laundi",
    "chirkut","tapori","gawar","junglee","kutta sala","haram ka pilla",
    "pkmkb","bkl","bklol",
]
PROFANITY_LIST = EN_PROFANITY_LIST + HI_PROFANITY_LIST

DEVANAGARI_ABUSE_MAP = {
    re.compile(r"मादरचोद", re.IGNORECASE): "madarchod",
    re.compile(r"भेनचोद|बहनचोद", re.IGNORECASE): "behenchod",
    re.compile(r"चूतिया|चुतिया", re.IGNORECASE): "chutiya",
    re.compile(r"गाण्ड|गांड", re.IGNORECASE): "gaand",
    re.compile(r"रंडी|रांड", re.IGNORECASE): "randi",
    re.compile(r"हरामी", re.IGNORECASE): "harami",
    re.compile(r"कमीना|कमीने", re.IGNORECASE): "kamina",
    re.compile(r"लौड़ा|लोडा|लंड", re.IGNORECASE): "lauda",
    re.compile(r"भोसड़ा|भोसड़ीके", re.IGNORECASE): "bhosdike",
    re.compile(r"साला|साली", re.IGNORECASE): "saala",
    re.compile(r"हरामजादा|हरामज़ादा", re.IGNORECASE): "haramzada",
    re.compile(r"कुत्ता|कुत्ते|कुतिया", re.IGNORECASE): "kutte",
    re.compile(r"हिजड़ा|हिजरा|चक्का", re.IGNORECASE): "hijra",
    re.compile(r"दल्ला|दलाल", re.IGNORECASE): "dalla",
}

SAFE_WORDS_EN = {
    "bass","mass","lass","glass","class","grass","pass","assume","asset","assess",
    "classic","assist","associate","scrap","ship","shot","shop","show","duck","luck",
    "muck","buck","truck","stuck","pluck","clock","block","flock","mock","rock","sock",
    "dock","witch","switch","stitch","pitch","hitch","ditch","rich","dam","damp","clam",
    "slam","tram","gram","bit","kit","sit","hit","fit","wit","pit","knit","can","ban",
    "tan","ran","man","fan","plan","clan","puss","fuss","bus","plus","thus","cuss",
    "cost","lost","frost","host","post","most","wick","nick","lick","tick","sick","pick","kick",
}
SAFE_WORDS_HI = {
    "lal","bal","kal","hal","mal","gal","dal","sal","gandh","gandha","gandhi",
    "saali","sala","maa","yaar","bhai","behen","beta","beti","dost",
    "kya","hai","nahi","haan","aur","par","se","ko","ka","ki","ke","ek","sab",
    "log","din","raat","ghar","kaam","baat","lad","ladi","ganda","chal","cha",
    "chai","chalo","chup","chori","kuch","koi","kahin","kahaan","kab","kyun",
    "kaisa","kaisi","theek","bahut","thoda","zyada","kam","achha","bura","sundar",
}
SAFE_WORDS = SAFE_WORDS_EN | SAFE_WORDS_HI

HINGLISH_SLANG_MAP: List[Tuple[re.Pattern, str]] = [
    (re.compile(r"\bky+a+\b", re.IGNORECASE), "kya"),
    (re.compile(r"\bn[ae]h[ii]+n?\b", re.IGNORECASE), "nahi"),
    (re.compile(r"\bnai\b", re.IGNORECASE), "nahi"),
    (re.compile(r"\bha+i+\b", re.IGNORECASE), "hai"),
    (re.compile(r"(?<!\w)h(?!\w)"), "hai"),
    (re.compile(r"\bya+r+\b", re.IGNORECASE), "yaar"),
    (re.compile(r"\bbha+i+\b", re.IGNORECASE), "bhai"),
    (re.compile(r"\bac+h+a\b", re.IGNORECASE), "achha"),
    (re.compile(r"\bokk*\b", re.IGNORECASE), "ok"),
    (re.compile(r"\b(?:ha){2,}\b", re.IGNORECASE), "haha"),
    (re.compile(r"\b(?:he){2,}\b", re.IGNORECASE), "hehe"),
    (re.compile(r"\blol+\b", re.IGNORECASE), "lol"),
    (re.compile(r"\blmao+\b", re.IGNORECASE), "lmao"),
    (re.compile(r"\bmtlb\b", re.IGNORECASE), "matlab"),
    (re.compile(r"\bmatla+b\b", re.IGNORECASE), "matlab"),
    (re.compile(r"\ba+ga+r\b", re.IGNORECASE), "agar"),
    (re.compile(r"\bphi+r\b", re.IGNORECASE), "phir"),
    (re.compile(r"\bfi+r\b", re.IGNORECASE), "phir"),
    (re.compile(r"\bab+hi+\b", re.IGNORECASE), "abhi"),
    (re.compile(r"\bavi\b", re.IGNORECASE), "abhi"),
    (re.compile(r"\bwai+se\b", re.IGNORECASE), "waise"),
    (re.compile(r"\bvaise\b", re.IGNORECASE), "waise"),
    (re.compile(r"\bto+h\b", re.IGNORECASE), "toh"),
    (re.compile(r"\bky[ou]+nki\b", re.IGNORECASE), "kyunki"),
    (re.compile(r"\bcoz\b", re.IGNORECASE), "kyunki"),
    (re.compile(r"\bthod+a\b", re.IGNORECASE), "thoda"),
    (re.compile(r"\bthodi\b", re.IGNORECASE), "thodi"),
    (re.compile(r"\bbilku+l\b", re.IGNORECASE), "bilkul"),
    (re.compile(r"\bthee+k\b", re.IGNORECASE), "theek"),
    (re.compile(r"\bteek\b", re.IGNORECASE), "theek"),
    (re.compile(r"\bthik\b", re.IGNORECASE), "theek"),
    (re.compile(r"\b(?:lm)?ao+\b", re.IGNORECASE), "lmao"),
    (re.compile(r"\bkek+\b", re.IGNORECASE), "lol"),
    (re.compile(r"\bxd+\b", re.IGNORECASE), "lol"),
]

_SPECIAL = r'[*!@#$%^&~0]'

def _word_to_masked_regex(word: str) -> str:
    if len(word) <= 2: return re.escape(word)
    if ' ' in word: return r'\s+'.join(_word_to_masked_regex(w) for w in word.split())
    first, *middle, last = word
    mid_pat = ''.join(rf'(?:{re.escape(ch)}|{_SPECIAL})+' for ch in middle)
    return rf'\b{re.escape(first)}+{mid_pat}{re.escape(last)}+\b'

def build_profanity_patterns(profanity_list):
    return [(re.compile(_word_to_masked_regex(w), re.IGNORECASE), w) for w in profanity_list]

_PROFANITY_PATTERNS = build_profanity_patterns(PROFANITY_LIST)

# ── Script detection ───────────────────────────────────────────────────────
def detect_script(text: str) -> str:
    deva_count, latin_count = 0, 0
    for ch in text:
        name = unicodedata.name(ch, '')
        if 'DEVANAGARI' in name: deva_count += 1
        elif ch.isalpha() and ord(ch) < 128: latin_count += 1
    if deva_count > 0 and deva_count >= latin_count * 0.4: return 'devanagari'
    hinglish_markers = re.compile(
        r'\b(maa|bhai|yaar|teri|tere|kya|hai|nahi|kar|ki|ke|ka|aur|'
        r'bhi|toh|se|ne|ko|par|ek|do|teen|acha|haan|nahi|kuch)\b', re.IGNORECASE)
    if hinglish_markers.search(text): return 'hinglish'
    return 'english'

# ── Pass-level helpers ─────────────────────────────────────────────────────
def _normalize_devanagari_abuse(text):
    for pat, rep in DEVANAGARI_ABUSE_MAP.items(): text = pat.sub(rep, text)
    return text

def _normalize_censored_words(text, patterns):
    for pat, rep in patterns: text = pat.sub(rep, text)
    return text

def _normalize_hinglish_slang(text):
    for pat, rep in HINGLISH_SLANG_MAP: text = pat.sub(rep, text)
    return text

def _normalize_repeated_chars(text, profanity_list, safe_words):
    def _thresh(w): n=len(w); return 0.15 if n<=4 else 0.20 if n<=6 else 0.25 if n<=9 else 0.30
    def process(word):
        if re.search(r'[\u0900-\u097F]', word): return word
        cleaned = re.sub(r'[^a-z]', '', word.lower())
        if not cleaned or len(cleaned) < 3 or cleaned in safe_words: return word
        reduced = re.sub(r'(.)\1{2,}', r'\1\1', cleaned)
        if reduced in safe_words: return word
        if not HAS_LEV: return word
        best, md = None, float('inf')
        for pw in profanity_list:
            cw2 = pw.replace(' ', '')
            d = lev_distance(reduced, cw2)
            if d < md: md, best = d, pw
        if best is None: return word
        rel = md / max(len(best.replace(' ', '')), 1)
        if (md <= 1 or rel <= _thresh(best.replace(' ', ''))) and reduced not in safe_words and cleaned not in safe_words:
            return best
        return word
    return ' '.join(process(w) for w in text.split())

# ── Pipeline: Hinglish ─────────────────────────────────────────────────────
def normalize_hinglish(text: str) -> str:
    text = text.lower()
    if HAS_EMOJI: text = _emoji_norm.replace_emoji(text, replace='')
    text = re.sub(r'https?\S+|www\S+', '', text)
    text = _normalize_devanagari_abuse(text)
    text = _normalize_censored_words(text, _PROFANITY_PATTERNS)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = _normalize_hinglish_slang(text)
    text = _normalize_repeated_chars(text, PROFANITY_LIST, SAFE_WORDS)
    text = re.sub(r'[^\w\u0900-\u097F\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ── Pipeline: English ──────────────────────────────────────────────────────
ENGLISH_CONTRACTIONS = {
    "won't": "will not", "can't": "cannot", "n't": " not",
    "'re": " are", "'ve": " have", "'ll": " will",
    "'d": " would", "'m": " am", "it's": "it is",
}
def normalize_english(text: str) -> str:
    if not isinstance(text, str): return ''
    if HAS_FTFY: text = ftfy.fix_text(text)
    if HAS_EMOJI:
        text = _emoji_norm.demojize(text, delimiters=(' ', ' '))
        text = re.sub(r'_+', ' ', text)
    text = text.lower()
    for c, e in ENGLISH_CONTRACTIONS.items(): text = text.replace(c, e)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'[^\w\s!?.,\'\-]', ' ', text, flags=re.UNICODE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ── Pipeline: Devanagari ───────────────────────────────────────────────────
def normalize_devanagari(text: str) -> str:
    if not isinstance(text, str): return ''
    if HAS_FTFY: text = ftfy.fix_text(text)
    if HAS_EMOJI:
        text = _emoji_norm.demojize(text, delimiters=(' ', ' '))
        text = re.sub(r'_+', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    if HAS_INDIC:
        text = _deva_normalizer.normalize(text)
    else:
        text = re.sub(r'[\u200c\u200d\u200b\u2060\ufeff]', '', text)
        text = re.sub(r'।{2,}', '।', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ── Auto-dispatch ──────────────────────────────────────────────────────────
def normalize(text: str, script: Optional[str] = None) -> Tuple[str, str]:
    """Auto-detect script and apply correct pipeline. Returns (norm_text, script)."""
    if script is None: script = detect_script(str(text))
    if script == 'devanagari': return normalize_devanagari(text), script
    if script == 'english':    return normalize_english(text), script
    return normalize_hinglish(text), script   # hinglish default

print('✅ Normalization pipelines ready (Hinglish | English | Devanagari).')


[WARN] indic-nlp-library not found — Devanagari uses regex fallback.
✅ Normalization pipelines ready (Hinglish | English | Devanagari).


In [6]:
!pip install python-Levenshtein
!pip install joblib
!pip install tqdm
!pip install emoji
!pip install ftfy
!pip install better_profanity

## Cell 2 — Imports

In [7]:
# =========================================
# IMPORTS + SETUP
# =========================================
import os, gc, warnings, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import emoji as emoji_lib

warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import Dataset, DataLoader
# from transformers import AutoTokenizer, AutoModelForSequenceClassification , get_linear_schedule_with_warmup
from torch.optim import AdamW

# =========================================
# DEVICE + SEED
# =========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

print("Device:", device)


Device: cuda


## Cell 3 — Configuration
**Change only here — nothing is hardcoded elsewhere.**

In [8]:
# =========================================
# MODELS
# =========================================
MODELS = {
    'HingRoBERTa': 'l3cube-pune/hing-roberta',
    'XLM-RoBERTa': 'xlm-roberta-base',
}

# =========================================
# LABELS
# =========================================
NUM_LABELS  = 2
LABEL_NAMES = ['Non-Hate', 'Hate']   # 0, 1

# =========================================
# TRAINING HYPERPARAMETERS
# =========================================
EPOCHS         = 4          # 5 is okay, but 4 avoids overfitting on small data
BATCH_SIZE     = 16
LEARNING_RATE  = 2e-5
MAX_SEQ_LEN    = 128

WARMUP_RATIO   = 0.1
WEIGHT_DECAY   = 0.01
GRAD_CLIP      = 1.0

VAL_SIZE       = 0.10       # IMPORTANT: match your earlier pipeline (10%)
RANDOM_SEED    = 42

# =========================================
# XAI SETTINGS
# =========================================
OUTPUT_ATTENTIONS    = True
OUTPUT_HIDDEN_STATES = True

XAI_SAMPLE_SIZE      = 200   # keep small to avoid memory crash
AOPC_STEPS           = 10

# =========================================
# OUTPUT PATHS
# =========================================
BASE_DIR = BASE_DIR = '/content/mydrive/MyDrive/roberta_outputs'
MODEL_DIR = os.path.join(BASE_DIR, 'models')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
XAI_DIR = os.path.join(BASE_DIR, 'xai')

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(XAI_DIR, exist_ok=True)

# =========================================
# SEED
# =========================================
import random
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

print('✅ Config ready.')

✅ Config ready.


In [13]:
import os
print(os.getcwd())

/content


In [10]:
!ls /content/roberta_outputs

ls: cannot access '/content/roberta_outputs': No such file or directory


In [15]:
from google.colab import drive
drive.mount('/content/mydrive')

Drive already mounted at /content/mydrive; to attempt to forcibly remount, call drive.mount("/content/mydrive", force_remount=True).


In [11]:
print("MODEL_DIR exists:", os.path.exists(MODEL_DIR))
print("RESULTS_DIR exists:", os.path.exists(RESULTS_DIR))
print("XAI_DIR exists:", os.path.exists(XAI_DIR))

MODEL_DIR exists: True
RESULTS_DIR exists: True
XAI_DIR exists: True


## Cell 4 — Emoji Cleaning
Removes low-signal positive emojis and caps repetitions at 2 per emoji.
No text normalization applied (as per project spec).

In [9]:
# Emoji cleaning is now handled inside each normalization pipeline.
# clean_emojis() kept as a lightweight fallback for any raw pre-check.
import collections
import emoji as emoji_lib

def clean_emojis(text, max_per_emoji=2):
    text = str(text)
    cleaned = ''.join('' if emoji_lib.is_emoji(ch) else ch for ch in text)
    return re.sub(r'\s+', ' ', cleaned).strip()

print('clean_emojis() ready (fallback only — normalization pipelines handle emoji).')


clean_emojis() ready (fallback only — normalization pipelines handle emoji).


## Cell 5 — Upload & Load Datasets

In [13]:
from google.colab import files
print('Upload 1/3 — FINAL_NLP_DATASET.csv (Hinglish TRAINING)')
u1 = files.upload()
hinglish_file = list(u1.keys())[0]

print('Upload 2/3 — cleaned_final_english.csv (English TEST)')
u2 = files.upload()
english_file = list(u2.keys())[0]

print('Upload 3/3 — cleaned_hindi_final.csv (Hindi TEST)')
u3 = files.upload()
hindi_file = list(u3.keys())[0]


Upload 1/3 — FINAL_NLP_DATASET.csv (Hinglish TRAINING)


Saving cleaned_hinglish.csv to cleaned_hinglish (1).csv
Upload 2/3 — cleaned_final_english.csv (English TEST)


Saving cleaned_english.csv to cleaned_english (3).csv
Upload 3/3 — cleaned_hindi_final.csv (Hindi TEST)


Saving cleaned_hindi.csv to cleaned_hindi (2).csv


In [14]:
def load_data(path, text_col, label_col, name):
    df = pd.read_csv(path)
    df = df[[text_col, label_col]].dropna().reset_index(drop=True)

    df['text']  = df[text_col].astype(str)
    df['label'] = df[label_col].astype(int)

    df = df[['text', 'label']]

    print(f"{name}: {len(df)} rows")

    return df

In [15]:
# =========================================
# LOAD DATASETS
# =========================================
train_df = load_data(hinglish_file, 'text', 'label', 'Hinglish')
english_df = load_data(english_file, 'text', 'label', 'English')
hindi_df = load_data(hindi_file, 'text', 'label', 'Hindi')

# =========================================
# APPLY NORMALIZATION — per source script
# =========================================
# Training data is Hinglish → use Hinglish pipeline directly
print('Normalizing Hinglish training set...')
train_df['text'] = train_df['text'].apply(normalize_hinglish)

# English test set → use English pipeline directly
print('Normalizing English test set...')
english_df['text'] = english_df['text'].apply(normalize_english)

# Hindi test set → use Devanagari pipeline directly
print('Normalizing Hindi (Devanagari) test set...')
hindi_df['text'] = hindi_df['text'].apply(normalize_devanagari)

print('✅ All datasets normalized.')
print(f'  Train (Hinglish): {len(train_df):,} rows')
print(f'  English test    : {len(english_df):,} rows')
print(f'  Hindi test      : {len(hindi_df):,} rows')

# Quick sanity check — show 3 samples per set
print('\n--- Sample normalized texts ---')
for label, df in [('Hinglish', train_df), ('English', english_df), ('Hindi', hindi_df)]:
    print(f'\n[{label}]')
    for t in df['text'].sample(3, random_state=42).tolist():
        print(' ', t[:120])


Hinglish: 33984 rows
English: 24781 rows
Hindi: 8192 rows
Normalizing Hinglish training set...
Normalizing English test set...
Normalizing Hindi (Devanagari) test set...
✅ All datasets normalized.
  Train (Hinglish): 33,984 rows
  English test    : 24,781 rows
  Hindi test      : 8,192 rows

--- Sample normalized texts ---

[Hinglish]
  main bhi ek muslim main chahta hun waqt board per kaarvayi ho ine logon ne jo jameen ki usse jyada tune masjid banaya ut
  voter read it mut
  rhne dijiye sir abhi bachcha hai rathee

[English]
  i was with a bitch with a mustash for a year and a half? wtf is wrong itch me
  do not be a bitch.
  bitches tweets be like i wanna suck some dick then be like chill it is just a song! bitch what song is

[Hindi]
  कोरोना दुनिया में: जापान के वैज्ञानिकों ने अर्थराइटिस की दवा एक्टेमरा से इलाज का दावा किया, यहां 67 हजार से ज्यादा संक्र
  यदि कोई अभिनेता की हत्या हो जाती तोह भारतीय मीडिया हाय हाय करती परन्तु साधु की हत्या पर कितने दिन न्यूज चलाई याद रखना सा
  Band

## Cell 6 — PyTorch Dataset & DataLoader

In [16]:
class HateSpeechDataset(Dataset):
    """Tokenized dataset. Keeps raw text alongside tensors for XAI reference."""
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            **{k: v.squeeze(0) for k, v in enc.items()},
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
            'raw_text': self.texts[idx],
        }


def xai_collate_fn(batch):
    """Handles the raw_text string field cleanly."""
    raw_texts = [b.pop('raw_text') for b in batch]
    labels    = torch.stack([b.pop('labels') for b in batch])
    keys      = list(batch[0].keys())
    tensors   = {k: torch.stack([b[k] for b in batch]) for k in keys}
    tensors['labels']    = labels
    tensors['raw_texts'] = raw_texts
    return tensors #data format handeling


def make_loader(texts, labels, tokenizer, batch_size, max_len, shuffle):
    ds = HateSpeechDataset(texts, labels, tokenizer, max_len)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=True,
                      collate_fn=xai_collate_fn) #for batching and shuffling

print('Dataset class ready.')

Dataset class ready.


## Cell 7 — Training & Evaluation Functions

testing returns macro f1 and accuracy and label predictions+ their probabilities

In [17]:
# def run_train_epoch(model, loader, optimizer, scheduler, loss_fn, device):
#     model.train()
#     total_loss, correct, total = 0.0, 0, 0
#     for batch in loader:
#         batch.pop('raw_texts')
#         labels = batch.pop('labels').to(device)
#         inputs = {k: v.to(device) for k, v in batch.items()}
#         optimizer.zero_grad()
#         # During training: no attentions/hidden states (saves memory)
#         out  = model(**inputs, output_attentions=False, output_hidden_states=False)
#         loss = loss_fn(out.logits, labels)
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
#         optimizer.step()
#         scheduler.step()
#         total_loss += loss.item()
#         correct    += (out.logits.argmax(-1) == labels).sum().item()
#         total      += labels.size(0)
#     return total_loss / len(loader), correct / total


def run_eval_epoch(model, loader, loss_fn, device):
    model.eval()
    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []
    with torch.no_grad():
        for batch in loader:
            batch.pop('raw_texts')
            labels = batch.pop('labels').to(device)
            inputs = {k: v.to(device) for k, v in batch.items()}
            out    = model(**inputs, output_attentions=False, output_hidden_states=False)
            loss   = loss_fn(out.logits, labels)
            probs  = torch.softmax(out.logits, dim=-1)
            total_loss += loss.item()
            all_preds.extend(out.logits.argmax(-1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            all_probs.extend(probs.cpu().tolist())
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    accuracy = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), macro_f1, accuracy, all_preds, all_labels, all_probs

print('Training functions ready.')

Training functions ready.


## Cell 8 — Train Both Models on Hinglish
HingRoBERTa → XLM-RoBERTa sequentially.
Each model saves its best checkpoint (by Val Macro F1).

In [18]:
# =========================================
# STRATIFIED TRAIN / VALIDATION SPLIT
# =========================================
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    train_df['text'],
    train_df['label'],
    test_size=VAL_SIZE,          # use your config
    stratify=train_df['label'],
    random_state=RANDOM_SEED
)

print(f"Train: {len(X_tr):,} | Val: {len(X_val):,}")

# =========================================
# CLASS WEIGHTS (HANDLE IMBALANCE)
# =========================================
import numpy as np
import torch

class_counts = np.bincount(y_tr)

# safety: avoid division by zero
class_counts = np.where(class_counts == 0, 1, class_counts)

cw = torch.tensor([1.0 / c for c in class_counts], dtype=torch.float)

# normalize weights
cw = cw / cw.sum() * NUM_LABELS
cw = cw.to(device)

print(f"Class weights → Non-Hate: {cw[0]:.3f} | Hate: {cw[1]:.3f}")

# =========================================
# LOSS FUNCTION (APPLY CLASS WEIGHTS)
# =========================================
loss_fn = torch.nn.CrossEntropyLoss(weight=cw)

# =========================================
# STORAGE DICTIONARIES
# =========================================
trained_models = {}     # {model_name: {best_path, tokenizer, best_f1}}
all_test_results = {}   # {model_name: {dataset: metrics}}

print("✅ Split + class weights ready.")

Train: 30,585 | Val: 3,399
Class weights → Non-Hate: 0.571 | Hate: 1.429
✅ Split + class weights ready.


In [ ]:
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm

# 🔥 UPDATED TRAIN FUNCTION (ONLY CHANGE)
def run_train_epoch(model, loader, optimizer, scheduler, loss_fn, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    loader = tqdm(loader, desc="Training", leave=False)  # 👈 progress bar

    for batch in loader:
        batch.pop('raw_texts')
        labels = batch.pop('labels').to(device)
        inputs = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        out  = model(**inputs, output_attentions=False, output_hidden_states=False)
        loss = loss_fn(out.logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        correct    += (out.logits.argmax(-1) == labels).sum().item()
        total      += labels.size(0)

        loader.set_postfix(loss=loss.item())  # 👈 live loss

    return total_loss / len(loader), correct / total  #accuracy and loss


# =========================================
# MAIN TRAINING LOOP (UNCHANGED)
# =========================================
for model_tag, model_name in MODELS.items():
    print(f'\n{"="*65}')
    print(f'  TRAINING: {model_tag}')
    print(f'  HuggingFace: {model_name}')
    print(f'{"="*65}')

    model_dir = os.path.join(BASE_DIR, model_tag.replace('-', '_').replace(' ', '_'))
    best_ckpt = os.path.join(model_dir, 'best_checkpoint')
    os.makedirs(model_dir, exist_ok=True)

    # TOKENIZER
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # LOADERS
    tr_loader  = make_loader(X_tr,  y_tr,  tokenizer, BATCH_SIZE, MAX_SEQ_LEN, shuffle=True)
    val_loader = make_loader(X_val, y_val, tokenizer, BATCH_SIZE, MAX_SEQ_LEN, shuffle=False)

    # MODEL
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=NUM_LABELS
    ).to(device)

    # OPTIMIZER + SCHEDULER
    no_decay = ['bias', 'LayerNorm.weight']

    opt_params = [
        {
            'params': [p for n, p in model.named_parameters()
                       if not any(nd in n for nd in no_decay)],
            'weight_decay': WEIGHT_DECAY
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if any(nd in n for nd in no_decay)],
            'weight_decay': 0.0
        }
    ]

    optimizer = AdamW(opt_params, lr=LEARNING_RATE)

    total_steps  = max(1, len(tr_loader) * EPOCHS)
    warmup_steps = int(WARMUP_RATIO * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    loss_fn = torch.nn.CrossEntropyLoss(weight=cw)

    # TRAIN LOOP
    history    = {'tr_loss':[], 'tr_acc':[], 'val_loss':[], 'val_f1':[], 'val_acc':[]}
    best_f1    = 0.0
    best_epoch = 0

    print(f'\n{"Ep":<5}{"TrLoss":<11}{"TrAcc":<11}{"VaLoss":<11}{"VaF1":<10}{"VaAcc"}')
    print('-' * 58)

    for ep in range(1, EPOCHS + 1):

        print(f"\n🔄 Running Epoch {ep}/{EPOCHS}...")  # 👈 added status

        tl, ta = run_train_epoch(
            model, tr_loader, optimizer, scheduler, loss_fn, device
        )

        vl, vf, va, _, _, _ = run_eval_epoch(
            model, val_loader, loss_fn, device
        )

        history['tr_loss'].append(tl)
        history['tr_acc'].append(ta)
        history['val_loss'].append(vl)
        history['val_f1'].append(vf)
        history['val_acc'].append(va)

        flag = ' ← best' if vf > best_f1 else ''
        print(f'{ep:<5}{tl:<11.4f}{ta:<11.4f}{vl:<11.4f}{vf:<10.4f}{va:.4f}{flag}')

        if vf > best_f1:
            best_f1 = vf
            best_epoch = ep

            model.save_pretrained(best_ckpt)
            tokenizer.save_pretrained(best_ckpt)

    print(f'\nBest: Epoch {best_epoch}  |  Val Macro F1 = {best_f1:.4f}')
    print(f'Checkpoint saved → {best_ckpt}/')

    trained_models[model_tag] = {
        'best_path':   best_ckpt,
        'tokenizer':   tokenizer,
        'history':     history,
        'best_val_f1': best_f1,
        'best_epoch':  best_epoch,
        'model_dir':   model_dir,
    }

    # CLEAN MEMORY
    del model
    gc.collect()
    torch.cuda.empty_cache()

print('\n✅ Both models trained and saved.')


  TRAINING: HingRoBERTa
  HuggingFace: l3cube-pune/hing-roberta


tokenizer_config.json:   0%|          | 0.00/406 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/750 [00:00<?, ?B/s]

...


Ep   TrLoss     TrAcc      VaLoss     VaF1      VaAcc
----------------------------------------------------------

🔄 Running Epoch 1/4...


1    0.5144     0.7906     0.3655     0.8293    0.8503 ← best

🔄 Running Epoch 2/4...


2    0.3446     0.8851     0.3058     0.8899    0.9082 ← best

🔄 Running Epoch 3/4...


3    0.3005     0.9063     0.3170     0.8858    0.9035

🔄 Running Epoch 4/4...


4    0.2608     0.9188     0.3518     0.8878    0.9059

Best: Epoch 2  |  Val Macro F1 = 0.8899
Checkpoint saved → roberta_outputs/XLM_RoBERTa/best_checkpoint/

✅ Both models trained and saved.


In [ ]:
# =========================================
# PLOT TRAINING CURVES (LOSS + ACCURACY)
# =========================================
eps = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# -------- LOSS --------
axes[0].plot(eps, history['tr_loss'], marker='o', label='Train Loss')
axes[0].plot(eps, history['val_loss'], marker='o', label='Val Loss')
axes[0].set_title('Loss vs Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid()

# -------- ACCURACY --------
axes[1].plot(eps, history['tr_acc'], marker='o', label='Train Acc')
axes[1].plot(eps, history['val_acc'], marker='o', label='Val Acc')
axes[1].set_title('Accuracy vs Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid()

plt.suptitle(f'{model_tag} Training Performance', fontsize=14)
plt.tight_layout()

# SAVE GRAPH
graph_path = os.path.join(model_dir, 'training_graphs.png')
plt.savefig(graph_path, dpi=150)
plt.close()

print(f"📊 Training graphs saved → {graph_path}")

📊 Training graphs saved → roberta_outputs/XLM_RoBERTa/training_graphs.png


In [19]:
# =========================================
# CREATE TEST DATASETS
# =========================================
combined_df = pd.concat([
    pd.DataFrame({'text': X_val, 'label': y_val}),
    english_df,
    hindi_df
]).reset_index(drop=True)

TEST_SETS = {
    'hinglish_test': pd.DataFrame({'text': X_val, 'label': y_val}),
    'english': english_df,
    'hindi': hindi_df,
    'combined': combined_df
}

## Cell 9 — Cross-Lingual Testing
Evaluates each best checkpoint on English test set, then Hindi test set.
Saves per-sample predictions for XAI.

In [ ]:
# =========================================
# CREATE TEST DATASETS
# =========================================
combined_df = pd.concat([
    pd.DataFrame({'text': X_val, 'label': y_val}),
    english_df,
    hindi_df
]).reset_index(drop=True)

TEST_SETS = {
    'hinglish_test': pd.DataFrame({'text': X_val, 'label': y_val}),
    'english': english_df,
    'hindi': hindi_df,
    'combined': combined_df
}

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

# =========================================
# STORE RESULTS
# =========================================
test_results = {}

# =========================================
# TEST BOTH MODELS
# =========================================
for model_tag, info in trained_models.items():

    print(f'\n{"="*65}')
    print(f'🚀 TESTING MODEL: {model_tag}')
    print(f'{"="*65}')

    model = AutoModelForSequenceClassification.from_pretrained(
        info['best_path']
    ).to(device)

    tokenizer = info['tokenizer']
    model.eval()

    test_results[model_tag] = {}

    for split_name, df in TEST_SETS.items():

        print(f'\n→ {model_tag} on {split_name}')

        loader = make_loader(
            df['text'],
            df['label'],
            tokenizer,
            BATCH_SIZE,
            MAX_SEQ_LEN,
            shuffle=False
        )

        loss_fn = torch.nn.CrossEntropyLoss(weight=cw)

        _, f1, acc, preds, labels, _ = run_eval_epoch(
            model, loader, loss_fn, device
        )

        precision = precision_score(labels, preds, average='macro', zero_division=0)
        recall    = recall_score(labels, preds, average='macro', zero_division=0)

        # PRINT METRICS
        print(f'Accuracy  : {acc:.4f}')
        print(f'Macro F1  : {f1:.4f}')
        print(f'Precision : {precision:.4f}')
        print(f'Recall    : {recall:.4f}')

        print("\nClassification Report:")
        print(classification_report(labels, preds, target_names=LABEL_NAMES, zero_division=0))

        # STORE RESULTS
        test_results[model_tag][split_name] = {
            'accuracy': acc,
            'f1': f1,
            'precision': precision,
            'recall': recall
        }

        # =========================================
        # CONFUSION MATRIX
        # =========================================
        cm = confusion_matrix(labels, preds)

        fig, ax = plt.subplots(figsize=(5, 5))
        ConfusionMatrixDisplay(cm, display_labels=LABEL_NAMES).plot(
            ax=ax, cmap='Blues', colorbar=False
        )

        ax.set_title(f'{model_tag} — {split_name}')
        plt.tight_layout()

        save_path = os.path.join(info['model_dir'], f'cm_{split_name}.png')
        plt.savefig(save_path, dpi=150)
        plt.close()

        print(f'Confusion matrix saved → {save_path}')

    del model
    gc.collect()
    torch.cuda.empty_cache()

print("\n✅ Both models tested with full metrics.")


# =========================================
# TEST METRICS GRAPHS
# =========================================
import matplotlib.pyplot as plt

for dataset in TEST_SETS.keys():

    labels = []
    accs, f1s, precs, recs = [], [], [], []

    for model_tag in test_results:
        if dataset not in test_results[model_tag]:
            continue

        labels.append(model_tag)
        accs.append(test_results[model_tag][dataset]['accuracy'])
        f1s.append(test_results[model_tag][dataset]['f1'])
        precs.append(test_results[model_tag][dataset]['precision'])
        recs.append(test_results[model_tag][dataset]['recall'])

    x = range(len(labels))

    plt.figure(figsize=(8,5))

    plt.plot(x, accs, marker='o', label='Accuracy')
    plt.plot(x, f1s, marker='o', label='F1')
    plt.plot(x, precs, marker='o', label='Precision')
    plt.plot(x, recs, marker='o', label='Recall')

    plt.xticks(x, labels)
    plt.title(f'Model Comparison on {dataset}')
    plt.xlabel('Models')
    plt.ylabel('Score')
    plt.legend()
    plt.grid()

    save_path = os.path.join(BASE_DIR, f'test_metrics_{dataset}.png')
    plt.savefig(save_path, dpi=150)
    plt.close()

    print(f"📊 Test graph saved → {save_path}")


🚀 TESTING MODEL: HingRoBERTa

→ HingRoBERTa on hinglish_test
Accuracy  : 0.9150
Macro F1  : 0.8983
Precision : 0.8899
Recall    : 0.9083

Classification Report:
              precision    recall  f1-score   support

    Non-Hate       0.96      0.92      0.94      2428
        Hate       0.82      0.89      0.86       971

    accuracy                           0.91      3399
   macro avg       0.89      0.91      0.90      3399
weighted avg       0.92      0.91      0.92      3399

Confusion matrix saved → roberta_outputs/HingRoBERTa/cm_hinglish_test.png

→ HingRoBERTa on english
Accuracy  : 0.3367
Macro F1  : 0.3367
Precision : 0.5954
Recall    : 0.5973

Classification Report:
              precision    recall  f1-score   support

    Non-Hate       0.20      0.99      0.33      4162
        Hate       0.99      0.20      0.34     20619

    accuracy                           0.34     24781
   macro avg       0.60      0.60      0.34     24781
weighted avg       0.86      0.34      

## Cell 10 — XAI Output Generation
For each model × each test set, saves a stratified sample of XAI-ready arrays:
- `cls_attn_row` — CLS token attention over all tokens → Attention heatmaps
- `token_ranking` — tokens ranked by CLS attention → AOPC input
- `cls_hidden` — CLS hidden state → SHAP feature space
- `last_layer_attn` — full last-layer attention matrix → attention viz
- `all_layers_attn` — all-layer averaged attention → attention rollout
- `logits`, `probs` — model confidence → AOPC probability drop

In [ ]:
def generate_xai_outputs(model, tokenizer, texts, labels, model_tag,
                          split_name, model_dir, device,
                          n_samples=200):

    import os
    import numpy as np
    import pandas as pd
    import torch

    xai_dir = os.path.join(model_dir, f'xai_{split_name}')
    os.makedirs(xai_dir, exist_ok=True)

    # Stratified sampling
    df_tmp = pd.DataFrame({'text': texts, 'label': labels})
    n_per_class = n_samples // 2

    sample_df = (
        df_tmp.groupby('label', group_keys=False)
        .apply(lambda x: x.sample(min(len(x), n_per_class), random_state=42))
        .reset_index(drop=True)
    )

    print(f'    XAI sample: {len(sample_df)} rows')

    model.eval()
    records = []

    for _, row in sample_df.iterrows():
        text = row['text']
        label = int(row['label'])

        enc = tokenizer(
            text,
            max_length=128,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        enc_gpu = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            out = model(
                **enc_gpu,
                output_attentions=True,
                output_hidden_states=True
            )

        tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
        input_ids = enc['input_ids'][0].cpu().numpy()

        logits = out.logits[0].cpu().numpy()
        probs = torch.softmax(out.logits, dim=-1)[0].cpu().numpy()
        pred_label = int(np.argmax(logits))

        # Attention
        last_attn = out.attentions[-1][0].cpu().numpy()
        avg_attn = last_attn.mean(axis=0)
        cls_attn_row = avg_attn[0]

        # Hidden
        cls_hidden = out.hidden_states[-1][0, 0, :].cpu().numpy()

        records.append({
            'text': text,
            'true_label': label,
            'pred_label': pred_label,
            'tokens': tokens,
            'input_ids': input_ids,
            'logits': logits,
            'probs': probs,
            'cls_attn_row': cls_attn_row,
            'cls_hidden': cls_hidden
        })

    # Save main file
    np.savez(
        os.path.join(xai_dir, 'xai_samples.npz'),
        texts=np.array([r['text'] for r in records], dtype=object),
        true_labels=np.array([r['true_label'] for r in records]),
        pred_labels=np.array([r['pred_label'] for r in records]),
        logits=np.array([r['logits'] for r in records]),
        probs=np.array([r['probs'] for r in records]),
        cls_attn_rows=np.array([r['cls_attn_row'] for r in records]),
        cls_hiddens=np.array([r['cls_hidden'] for r in records]),
        allow_pickle=True
    )

    print(f'    Saved → {xai_dir}/')
    return records

In [ ]:
# =========================================
# XAI CONFIG
# =========================================
RUN_XAI_ON = ['train_sample', 'hinglish_test', 'english', 'hindi']

# =========================================
# CREATE DATASETS FOR XAI
# =========================================
train_sample_df = pd.DataFrame({
    'text': X_tr,
    'label': y_tr
}).sample(n=min(XAI_SAMPLE_SIZE, len(X_tr)), random_state=RANDOM_SEED)

hinglish_test_df = pd.DataFrame({
    'text': X_val,
    'label': y_val
})

combined_df = pd.concat([
    hinglish_test_df,
    english_df,
    hindi_df
]).reset_index(drop=True)

XAI_DATASETS = {
    'train_sample': train_sample_df,
    'hinglish_test': hinglish_test_df,
    'english': english_df,
    'hindi': hindi_df,
    'combined': combined_df
}

# =========================================
# RUN XAI FOR BOTH MODELS
# =========================================
for model_tag, info in trained_models.items():

    print(f'\n{"="*65}')
    print(f'🔍 RUNNING XAI FOR: {model_tag}')
    print(f'{"="*65}')

    tokenizer = info['tokenizer']
    model_dir = info['model_dir']

    model = AutoModelForSequenceClassification.from_pretrained(
        info['best_path'],
        output_attentions=True,
        output_hidden_states=True
    ).to(device)

    model.eval()

    for split_name, df in XAI_DATASETS.items():

        if split_name not in RUN_XAI_ON:
            continue

        print(f'\n→ XAI on {split_name} ({len(df):,} samples)')

        generate_xai_outputs(
            model=model,
            tokenizer=tokenizer,
            texts=df['text'].tolist(),
            labels=df['label'].tolist(),
            model_tag=model_tag,
            split_name=split_name,
            model_dir=model_dir,
            device=device,
            n_samples=XAI_SAMPLE_SIZE
        )

    del model
    gc.collect()
    torch.cuda.empty_cache()

print("\n✅ XAI generation complete for both models (including Hindi).")


🔍 RUNNING XAI FOR: HingRoBERTa

→ XAI on train_sample (200 samples)
    XAI sample: 160 rows
    Saved → roberta_outputs/HingRoBERTa/xai_train_sample/

→ XAI on hinglish_test (3,399 samples)
    XAI sample: 200 rows
    Saved → roberta_outputs/HingRoBERTa/xai_hinglish_test/

→ XAI on english (24,781 samples)
    XAI sample: 200 rows
    Saved → roberta_outputs/HingRoBERTa/xai_english/

→ XAI on hindi (8,192 samples)
    XAI sample: 200 rows
    Saved → roberta_outputs/HingRoBERTa/xai_hindi/

🔍 RUNNING XAI FOR: XLM-RoBERTa

→ XAI on train_sample (200 samples)
    XAI sample: 160 rows
    Saved → roberta_outputs/XLM_RoBERTa/xai_train_sample/

→ XAI on hinglish_test (3,399 samples)
    XAI sample: 200 rows
    Saved → roberta_outputs/XLM_RoBERTa/xai_hinglish_test/

→ XAI on english (24,781 samples)
    XAI sample: 200 rows
    Saved → roberta_outputs/XLM_RoBERTa/xai_english/

→ XAI on hindi (8,192 samples)
    XAI sample: 200 rows
    Saved → roberta_outputs/XLM_RoBERTa/xai_hindi/

✅ XAI

## Cell 11 — AOPC (Attention-Based)
Computes the Area Over the Perturbation Curve using the pre-saved token rankings.
When SHAP rankings are available, pass them via `shap_rankings` to get SHAP-AOPC for comparison.

In [ ]:
def compute_aopc(model, tokenizer, records, device,
                 steps=AOPC_STEPS, shap_rankings=None, label='attention'):
    """
    AOPC: mask top-k tokens one-by-one, measure confidence drop.

    Args:
        records       : list of dicts from generate_xai_outputs()
        shap_rankings : optional List[List[int]] of SHAP-based token rankings
                        (replaces attention rankings if provided)
        label         : string tag for this run ('attention' or 'shap')
    Returns:
        dict: mean_aopc, all_scores (per sample), all_curves (per step)
    """
    model.eval()
    mask_token = tokenizer.mask_token or tokenizer.unk_token
    all_scores, all_curves = [], []

    for i, rec in enumerate(records):
        text         = rec['text']
        pred_class   = rec['pred_label']
        orig_prob    = float(rec['probs'][pred_class])

        # Choose ranking source
        if shap_rankings is not None:
            ranking = list(shap_rankings[i])
        else:
            ranking = list(rec['token_ranking'])

        words        = text.split()
        masked_words = words.copy()
        curve        = []

        def subtoken_to_word_idx(subtoken_idx):
            """Map subword token index → word index (approximate)."""
            real = int(subtoken_idx) - 1   # -1 for [CLS]
            if real < 0: return None
            cumulative = 0
            for w_idx, word in enumerate(words):
                cumulative += len(tokenizer.tokenize(word))
                if real < cumulative:
                    return w_idx
            return None

        for k in range(1, steps + 1):
            if k <= len(ranking):
                w_idx = subtoken_to_word_idx(ranking[k - 1])
                if w_idx is not None and w_idx < len(masked_words):
                    masked_words[w_idx] = mask_token

            perturbed = ' '.join(masked_words)
            enc = tokenizer(perturbed, return_tensors='pt',
                            max_length=MAX_SEQ_LEN, truncation=True,
                            padding='max_length')
            enc = {k2: v.to(device) for k2, v in enc.items()}
            with torch.no_grad():
                out = model(**enc, output_attentions=False, output_hidden_states=False)
            p = float(torch.softmax(out.logits, -1)[0][pred_class].cpu())
            curve.append(orig_prob - p)

        all_scores.append(float(np.mean(curve)))
        all_curves.append(curve)

    return {
        'label':      label,
        'mean_aopc':  float(np.mean(all_scores)),
        'std_aopc':   float(np.std(all_scores)),
        'all_scores': all_scores,
        'all_curves': all_curves,
    }

print('AOPC function ready.')

AOPC function ready.


In [ ]:
# =========================================
# AOPC RESULTS STORAGE
# =========================================
aopc_results_all = {}

for model_tag, info in trained_models.items():

    aopc_results_all[model_tag] = {}

    model_dir = info['model_dir']
    tokenizer = info['tokenizer']

    print(f'\n{"="*65}')
    print(f'  AOPC: {model_tag}')
    print(f'{"="*65}')

    # 🔥 Load model (LIGHT version)
    model = AutoModelForSequenceClassification.from_pretrained(
        info['best_path'],
        output_attentions=False,
        output_hidden_states=False
    ).to(device)

    model.eval()

    for split_name in ['english', 'hindi']:   # 🔥 limit for safety

        xai_dir = os.path.join(model_dir, f'xai_{split_name}')
        xai_file = os.path.join(xai_dir, 'xai_samples.npz')

        # ❗ Skip if XAI not present
        if not os.path.exists(xai_file):
            print(f"⚠️ Skipping {split_name} (no XAI file found)")
            continue

        print(f'\n  {split_name} — attention-based AOPC')

        # =========================================
        # LOAD XAI DATA (FROM DISK, NOT RAM)
        # =========================================
        data = np.load(xai_file, allow_pickle=True)

        records = []
        for i in range(len(data['texts'])):

            attn = data['cls_attn_rows'][i]

            # 🔥 create ranking (highest attention first)
            token_ranking = list(np.argsort(-attn))

            records.append({
                'text': data['texts'][i],
                'true_label': int(data['true_labels'][i]),
                'pred_label': int(data['pred_labels'][i]),
                'cls_attn_row': attn,
                'probs': data['probs'][i],
                'token_ranking': token_ranking
            })

        # =========================================
        # COMPUTE AOPC
        # =========================================
        attn_result = compute_aopc(
            model,
            tokenizer,
            records,
            device,
            label='attention'
        )

        print(f'    Mean AOPC: {attn_result["mean_aopc"]:.4f} ± {attn_result["std_aopc"]:.4f}')

        aopc_results_all[model_tag][split_name] = {
            'attention': attn_result
        }

        # =========================================
        # PLOT AOPC CURVE
        # =========================================
        mean_curve = np.mean(attn_result['all_curves'], axis=0)

        plt.figure(figsize=(7, 4))
        plt.plot(range(1, AOPC_STEPS + 1), mean_curve,
                 marker='o', color='darkorange')

        plt.xlabel('Tokens masked (k)')
        plt.ylabel('Mean probability drop')
        plt.title(f'AOPC — {model_tag} | {split_name}')
        plt.grid(alpha=0.3)
        plt.tight_layout()

        plt.savefig(os.path.join(xai_dir, 'aopc_attention.png'), dpi=150)
        plt.close()

        # =========================================
        # SAVE AOPC RESULTS
        # =========================================
        np.savez(
            os.path.join(xai_dir, 'aopc_attention.npz'),
            mean_aopc  = np.array(attn_result['mean_aopc']),
            std_aopc   = np.array(attn_result['std_aopc']),
            all_scores = np.array(attn_result['all_scores']),
            all_curves = np.array(attn_result['all_curves']),
        )

        print(f"    Saved → {xai_dir}/aopc_attention.*")

    # ✅ CORRECT POSITION (outside inner loop)
    del model
    gc.collect()
    torch.cuda.empty_cache()

print('\n✅ AOPC (attention) complete.')


  AOPC: HingRoBERTa

  english — attention-based AOPC
    Mean AOPC: 0.1073 ± 0.2774
    Saved → roberta_outputs/HingRoBERTa/xai_english/aopc_attention.*

  hindi — attention-based AOPC
    Mean AOPC: 0.0559 ± 0.1243
    Saved → roberta_outputs/HingRoBERTa/xai_hindi/aopc_attention.*

  AOPC: XLM-RoBERTa

  english — attention-based AOPC
    Mean AOPC: 0.0890 ± 0.2738
    Saved → roberta_outputs/XLM_RoBERTa/xai_english/aopc_attention.*

  hindi — attention-based AOPC
    Mean AOPC: -0.0150 ± 0.0734
    Saved → roberta_outputs/XLM_RoBERTa/xai_hindi/aopc_attention.*

✅ AOPC (attention) complete.


## Cell 12 — Results Summary Table

In [ ]:
rows = []

for model_tag, info in trained_models.items():
    for split_name in TEST_SETS:

        # 🔥 skip if test results not present
        if split_name not in test_results[model_tag]:
            continue

        test_res = test_results[model_tag][split_name]

        # 🔥 safe AOPC access
        if (model_tag in aopc_results_all and
            split_name in aopc_results_all[model_tag]):

            aopc_res = aopc_results_all[model_tag][split_name]['attention']
            aopc_val = round(aopc_res['mean_aopc'], 4)
        else:
            aopc_val = None   # or 'N/A'

        rows.append({
            'model':          model_tag,
            'train_data':     'Hinglish (normalized)',
            'test_data':      split_name,
            'normalization':  True,
            'best_val_f1':    round(info['best_val_f1'], 4),
            'best_epoch':     info['best_epoch'],
            'test_accuracy':  round(test_res['accuracy'], 4),
            'test_macro_f1':  round(test_res['f1'], 4),   # 🔥 fixed
            'test_precision': round(test_res['precision'], 4),
            'test_recall':    round(test_res['recall'], 4),
            'aopc_attention': aopc_val,
            'aopc_shap':      'TBD',
        })

# CREATE DF
summary_df = pd.DataFrame(rows)

# SAVE
summary_csv = os.path.join(BASE_DIR, 'full_results.csv')
summary_df.to_csv(summary_csv, index=False)

print('=== FULL RESULTS SUMMARY ===')
print(summary_df.to_string(index=False))

=== FULL RESULTS SUMMARY ===
      model            train_data     test_data  normalization  best_val_f1  best_epoch  test_accuracy  test_macro_f1  test_precision  test_recall  aopc_attention aopc_shap
HingRoBERTa Hinglish (normalized) hinglish_test           True       0.8983           2         0.9150         0.8983          0.8899       0.9083             NaN       TBD
HingRoBERTa Hinglish (normalized)       english           True       0.8983           2         0.3367         0.3367          0.5954       0.5973          0.1073       TBD
HingRoBERTa Hinglish (normalized)         hindi           True       0.8983           2         0.5403         0.5382          0.5382       0.5382          0.0559       TBD
HingRoBERTa Hinglish (normalized)      combined           True       0.8983           2         0.4366         0.4352          0.5475       0.5424             NaN       TBD
XLM-RoBERTa Hinglish (normalized) hinglish_test           True       0.8899           2         0.9082    

## Cell 13 — Download Everything

In [ ]:
import shutil
from google.colab import files

zip_name = 'roberta_outputs_backup'   # better name (avoid overwrite confusion)

print("📦 Creating zip...")

shutil.make_archive(zip_name, 'zip', BASE_DIR)

print(f"✅ Zipped: {zip_name}.zip")

files.download(f'{zip_name}.zip')

print("⬇️ Download triggered.")

📦 Creating zip...
✅ Zipped: roberta_outputs_backup.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️ Download triggered.


# THRESHOLD


In [2]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

path = '/content/mydrive/MyDrive/best_checkpoint_xlm'

model = AutoModelForSequenceClassification.from_pretrained(
    path,
    local_files_only=True
)

tokenizer = AutoTokenizer.from_pretrained(
    path,
    local_files_only=True
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [22]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

path2 = '/content/mydrive/MyDrive/best_checkpoint_hing'

model = AutoModelForSequenceClassification.from_pretrained(
    path2,
    local_files_only=True
)

tokenizer = AutoTokenizer.from_pretrained(
    path2,
    local_files_only=True
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [23]:
print(type(X_val), len(X_val))
print(type(TEST_SETS))
print(type(cw))
print(device)

<class 'pandas.core.series.Series'> 3399
<class 'dict'>
<class 'torch.Tensor'>
cuda


In [24]:
print("\n" + "="*70)
print("🔬 EXPERIMENTS: Threshold Optimization (Recovered Models)")
print("="*70)

import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score


# =========================================================
# 🔧 DEFINE YOUR SAVED MODEL PATHS HERE
# =========================================================
trained_models = {
    "XLM-RoBERTa": {
        "best_path": path
    },
    "HingRoBERTa": {
        "best_path": path2
    }
}


# =========================================================
# 🔍 THRESHOLD EXPERIMENT
# =========================================================
threshold_results = {}

for model_tag, info in trained_models.items():

    print(f"\n{'-'*60}")
    print(f"🔍 Threshold Experiment → {model_tag}")
    print(f"{'-'*60}")

    # ---------------------------
    # Load model + tokenizer
    # ---------------------------
    model = AutoModelForSequenceClassification.from_pretrained(
        info['best_path']
    ).to(device)

    tokenizer = AutoTokenizer.from_pretrained(info['best_path'])
    model.eval()

    # ---------------------------
    # Loss (same as training)
    # ---------------------------
    threshold_loss_fn = torch.nn.CrossEntropyLoss(weight=cw)

    # ---------------------------
    # VALIDATION → find best threshold
    # ---------------------------
    val_loader = make_loader(
        X_val, y_val, tokenizer,
        BATCH_SIZE, MAX_SEQ_LEN, shuffle=False
    )

    _, _, _, _, val_labels, val_probs = run_eval_epoch(
        model, val_loader, threshold_loss_fn, device
    )

    val_probs = np.array(val_probs)[:, 1]

    best_t = 0.5
    best_f1 = 0

    # 🔥 improved search
    for t in np.linspace(0.1, 0.9, 50):
        preds = (val_probs >= t).astype(int)
        f1 = f1_score(val_labels, preds, average='macro')

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    print(f"✅ Best Threshold (VAL): {best_t:.3f}")
    print(f"📊 Best VAL Macro-F1: {best_f1:.4f}")

    threshold_results[model_tag] = {}

    # =========================================================
    # TEST → apply best threshold
    # =========================================================
    for split_name, df in TEST_SETS.items():

        loader = make_loader(
            df['text'], df['label'],
            tokenizer, BATCH_SIZE, MAX_SEQ_LEN,
            shuffle=False
        )

        _, _, _, _, labels, probs = run_eval_epoch(
            model, loader, threshold_loss_fn, device
        )

        probs = np.array(probs)[:, 1]
        preds = (probs >= best_t).astype(int)

        acc  = accuracy_score(labels, preds)
        f1   = f1_score(labels, preds, average='macro')
        prec = precision_score(labels, preds, average='macro', zero_division=0)
        rec  = recall_score(labels, preds, average='macro', zero_division=0)

        print(f"\n📌 {split_name.upper()}")
        print(f"F1        : {f1:.4f}")
        print(f"Accuracy  : {acc:.4f}")
        print(f"Precision : {prec:.4f}")
        print(f"Recall    : {rec:.4f}")

        threshold_results[model_tag][split_name] = {
            'threshold': float(best_t),
            'f1': float(f1),
            'accuracy': float(acc),
            'precision': float(prec),
            'recall': float(rec)
        }

    # ---------------------------
    # Clean GPU memory
    # ---------------------------
    del model
    torch.cuda.empty_cache()


# =========================================================
# 📊 FINAL SUMMARY (VERY USEFUL FOR PAPER)
# =========================================================
print("\n" + "="*70)
print("📊 FINAL THRESHOLD RESULTS SUMMARY")
print("="*70)

for model_tag, splits in threshold_results.items():
    print(f"\n🔹 MODEL: {model_tag}")

    for split, metrics in splits.items():
        print(f"{split:15} → "
              f"F1: {metrics['f1']:.4f} | "
              f"Acc: {metrics['accuracy']:.4f} | "
              f"T: {metrics['threshold']:.2f}")


🔬 EXPERIMENTS: Threshold Optimization (Recovered Models)

------------------------------------------------------------
🔍 Threshold Experiment → XLM-RoBERTa
------------------------------------------------------------


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Best Threshold (VAL): 0.606
📊 Best VAL Macro-F1: 0.8931

📌 HINGLISH_TEST
F1        : 0.8931
Accuracy  : 0.9114
Precision : 0.8877
Recall    : 0.8991


KeyboardInterrupt: 

In [25]:
# ================================================================
# THRESHOLD OPTIMIZATION — tuned on Hindi val, reported on Hindi test
# ================================================================
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# ----------------------------------------------------------------
# Step 1: Split Hindi dataset into val (tuning) + test (reporting)
# ----------------------------------------------------------------
# Stratified so both halves keep the same class ratio
hindi_val_df, hindi_test_df = train_test_split(
    hindi_df,
    test_size=0.975,        # ~200 rows for val, rest for test
    random_state=RANDOM_SEED,
    stratify=hindi_df['label']
)

print(f"Hindi val  (threshold tuning) : {len(hindi_val_df)} rows")
print(f"Hindi test (final reporting)  : {len(hindi_test_df)} rows")
print(f"Val  class balance: {hindi_val_df['label'].value_counts().to_dict()}")
print(f"Test class balance: {hindi_test_df['label'].value_counts().to_dict()}")

# ----------------------------------------------------------------
# Step 2: Run threshold optimization only on HingRoBERTa
# ----------------------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(path2).to(device)
tokenizer = AutoTokenizer.from_pretrained(path2)
model.eval()

loss_fn = torch.nn.CrossEntropyLoss(weight=cw)

# --- get val probabilities ---
val_loader = make_loader(
    hindi_val_df['text'].tolist(),
    hindi_val_df['label'].tolist(),
    tokenizer, BATCH_SIZE, MAX_SEQ_LEN, shuffle=False
)
_, _, _, _, val_labels, val_probs = run_eval_epoch(model, val_loader, loss_fn, device)
val_probs = np.array(val_probs)[:, 1]   # probability of class 1 (Hate)

# --- grid search over thresholds ---
best_t, best_f1 = 0.5, 0.0

print("\nThreshold search (Hindi val):")
print(f"{'Threshold':>12} {'Macro-F1':>10}")
print("-" * 24)

for t in np.linspace(0.1, 0.9, 81):   # finer grid than before
    preds = (val_probs >= t).astype(int)
    f1 = f1_score(val_labels, preds, average='macro', zero_division=0)
    if f1 > best_f1:
        best_f1, best_t = f1, t

print(f"  → Best threshold : {best_t:.3f}")
print(f"  → Best val macro-F1 : {best_f1:.4f}")

# ----------------------------------------------------------------
# Step 3: Apply best threshold to held-out Hindi TEST set
# ----------------------------------------------------------------
test_loader = make_loader(
    hindi_test_df['text'].tolist(),
    hindi_test_df['label'].tolist(),
    tokenizer, BATCH_SIZE, MAX_SEQ_LEN, shuffle=False
)
_, _, _, _, test_labels, test_probs = run_eval_epoch(model, test_loader, loss_fn, device)
test_probs = np.array(test_probs)[:, 1]
test_preds = (test_probs >= best_t).astype(int)

acc  = accuracy_score(test_labels, test_preds)
f1   = f1_score(test_labels, test_preds, average='macro', zero_division=0)
prec = precision_score(test_labels, test_preds, average='macro', zero_division=0)
rec  = recall_score(test_labels, test_preds, average='macro', zero_division=0)

print("\n" + "="*50)
print("HINDI TEST — threshold-optimized results")
print("="*50)
print(f"Threshold  : {best_t:.3f}")
print(f"Accuracy   : {acc:.4f}")
print(f"Macro-F1   : {f1:.4f}")
print(f"Precision  : {prec:.4f}")
print(f"Recall     : {rec:.4f}")

# ----------------------------------------------------------------
# Step 4: Compare against default 0.5 threshold (sanity check)
# ----------------------------------------------------------------
default_preds = (test_probs >= 0.5).astype(int)
default_f1 = f1_score(test_labels, default_preds, average='macro', zero_division=0)
print(f"\nDefault (t=0.50) macro-F1 : {default_f1:.4f}")
print(f"Gain from optimization    : {f1 - default_f1:+.4f}")

del model
torch.cuda.empty_cache()

Hindi val  (threshold tuning) : 204 rows
Hindi test (final reporting)  : 7988 rows
Val  class balance: {0: 109, 1: 95}
Test class balance: {0: 4249, 1: 3739}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Threshold search (Hindi val):
   Threshold   Macro-F1
------------------------
  → Best threshold : 0.510
  → Best val macro-F1 : 0.5743

HINDI TEST — threshold-optimized results
Threshold  : 0.510
Accuracy   : 0.5499
Macro-F1   : 0.5454
Precision  : 0.5465
Recall     : 0.5458

Default (t=0.50) macro-F1 : 0.5481
Gain from optimization    : -0.0027


# FEW SHOT ON K=16

In [27]:


# ================================================================
# FEW-SHOT TRANSFER — HingRoBERTa fine-tuned on K Hindi examples
# ================================================================
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
import numpy as np
import torch

# ----------------------------------------------------------------
# Config — only change K to experiment (8, 16, 32)
# ----------------------------------------------------------------
FEW_SHOT_K     = 16    # examples PER CLASS (so 32 total for binary)
FEW_SHOT_EPOCHS = 3
FEW_SHOT_LR    = 5e-6  # much lower than training LR to avoid forgetting
FEW_SHOT_SEED  = RANDOM_SEED

# ----------------------------------------------------------------
# Step 1: Sample K examples per class from hindi_val_df
#         (the 204-row tuning split — never touch hindi_test_df)
# ----------------------------------------------------------------
few_shot_frames = []
for label in [0, 1]:
    subset = hindi_val_df[hindi_val_df['label'] == label]
    n = min(FEW_SHOT_K, len(subset))   # safety cap
    few_shot_frames.append(subset.sample(n=n, random_state=FEW_SHOT_SEED))

few_shot_df = pd.concat(few_shot_frames).sample(frac=1, random_state=FEW_SHOT_SEED).reset_index(drop=True)

print(f"Few-shot support set: {len(few_shot_df)} examples")
print(f"Class balance: {few_shot_df['label'].value_counts().to_dict()}")

# ----------------------------------------------------------------
# Step 2: Load checkpoint — start from HingRoBERTa best weights
# ----------------------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(path2).to(device)
tokenizer = AutoTokenizer.from_pretrained(path2)
model.train()

# ----------------------------------------------------------------
# Step 3: Fine-tune on few-shot support set
# ----------------------------------------------------------------
fs_loader = make_loader(
    few_shot_df['text'].tolist(),
    few_shot_df['label'].tolist(),
    tokenizer, 8, MAX_SEQ_LEN, shuffle=True   # batch=8 since dataset is tiny
)

optimizer = AdamW(model.parameters(), lr=FEW_SHOT_LR, weight_decay=0.01)
total_steps = len(fs_loader) * FEW_SHOT_EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,       # no warmup needed for 3 epochs on tiny data
    num_training_steps=total_steps
)

loss_fn = torch.nn.CrossEntropyLoss(weight=cw)

print(f"\nFine-tuning for {FEW_SHOT_EPOCHS} epochs on {len(few_shot_df)} examples...")
print(f"LR: {FEW_SHOT_LR} | Steps: {total_steps}")
print("-" * 40)

for epoch in range(FEW_SHOT_EPOCHS):
    model.train()
    epoch_loss = 0.0

    for batch in fs_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{FEW_SHOT_EPOCHS} — loss: {epoch_loss/len(fs_loader):.4f}")

# ----------------------------------------------------------------
# Step 4: Evaluate on hindi_test_df (the 7988-row held-out set)
# ----------------------------------------------------------------
model.eval()
test_loader = make_loader(
    hindi_test_df['text'].tolist(),
    hindi_test_df['label'].tolist(),
    tokenizer, BATCH_SIZE, MAX_SEQ_LEN, shuffle=False
)

_, _, _, _, test_labels, test_probs = run_eval_epoch(model, test_loader, loss_fn, device)
test_probs  = np.array(test_probs)[:, 1]
test_preds  = (test_probs >= 0.5).astype(int)

acc  = accuracy_score(test_labels, test_preds)
f1   = f1_score(test_labels, test_preds, average='macro', zero_division=0)
prec = precision_score(test_labels, test_preds, average='macro', zero_division=0)
rec  = recall_score(test_labels, test_preds, average='macro', zero_division=0)

print("\n" + "="*50)
print(f"FEW-SHOT (K={FEW_SHOT_K} per class) — Hindi test results")
print("="*50)
print(f"Accuracy  : {acc:.4f}")
print(f"Macro-F1  : {f1:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")

# Baseline comparison (from threshold experiment)
baseline_f1 = 0.5454
print(f"\nBaseline macro-F1     : {baseline_f1:.4f}")
print(f"Few-shot macro-F1     : {f1:.4f}")
print(f"Gain                  : {f1 - baseline_f1:+.4f}")

del model
torch.cuda.empty_cache()

Few-shot support set: 32 examples
Class balance: {1: 16, 0: 16}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Fine-tuning for 3 epochs on 32 examples...
LR: 5e-06 | Steps: 12
----------------------------------------
Epoch 1/3 — loss: 0.6852
Epoch 2/3 — loss: 0.6217
Epoch 3/3 — loss: 0.5483

FEW-SHOT (K=16 per class) — Hindi test results
Accuracy  : 0.5038
Macro-F1  : 0.3983
Precision : 0.6730
Recall    : 0.5327

Baseline macro-F1     : 0.5454
Few-shot macro-F1     : 0.3983
Gain                  : -0.1471


In [28]:
# ================================================================
# FEW-SHOT v2 — frozen encoder, only top 2 layers + head updated
# ================================================================

FEW_SHOT_K      = 16
FEW_SHOT_EPOCHS = 2      # fewer epochs
FEW_SHOT_LR     = 3e-5   # higher LR is okay since fewer params are updating
FEW_SHOT_SEED   = RANDOM_SEED

# --- support set (same as before) ---
few_shot_frames = []
for label in [0, 1]:
    subset = hindi_val_df[hindi_val_df['label'] == label]
    few_shot_frames.append(subset.sample(n=min(FEW_SHOT_K, len(subset)), random_state=FEW_SHOT_SEED))

few_shot_df = pd.concat(few_shot_frames).sample(frac=1, random_state=FEW_SHOT_SEED).reset_index(drop=True)
print(f"Support set: {len(few_shot_df)} | Balance: {few_shot_df['label'].value_counts().to_dict()}")

# --- load checkpoint ---
model = AutoModelForSequenceClassification.from_pretrained(path2).to(device)
tokenizer = AutoTokenizer.from_pretrained(path2)

# ----------------------------------------------------------------
# KEY CHANGE: freeze everything except top 2 encoder layers + head
# ----------------------------------------------------------------
# First freeze all parameters
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last 2 transformer layers
for layer in model.roberta.encoder.layer[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

# Unfreeze classifier head
for param in model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

# --- fine-tune ---
fs_loader = make_loader(
    few_shot_df['text'].tolist(),
    few_shot_df['label'].tolist(),
    tokenizer, 4, MAX_SEQ_LEN, shuffle=True   # batch=4 → more gradient steps
)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=FEW_SHOT_LR, weight_decay=0.01
)
loss_fn = torch.nn.CrossEntropyLoss(weight=cw)

print(f"\nFine-tuning {FEW_SHOT_EPOCHS} epochs | LR {FEW_SHOT_LR} | {len(fs_loader)} steps/epoch")
print("-" * 40)

for epoch in range(FEW_SHOT_EPOCHS):
    model.train()
    epoch_loss = 0.0
    for batch in fs_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        loss = loss_fn(model(input_ids=input_ids, attention_mask=attention_mask).logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{FEW_SHOT_EPOCHS} — loss: {epoch_loss/len(fs_loader):.4f}")

# --- evaluate ---
model.eval()
test_loader = make_loader(
    hindi_test_df['text'].tolist(),
    hindi_test_df['label'].tolist(),
    tokenizer, BATCH_SIZE, MAX_SEQ_LEN, shuffle=False
)

_, _, _, _, test_labels, test_probs = run_eval_epoch(model, test_loader, loss_fn, device)
test_preds = (np.array(test_probs)[:, 1] >= 0.5).astype(int)

acc  = accuracy_score(test_labels, test_preds)
f1   = f1_score(test_labels, test_preds, average='macro', zero_division=0)
prec = precision_score(test_labels, test_preds, average='macro', zero_division=0)
rec  = recall_score(test_labels, test_preds, average='macro', zero_division=0)

print("\n" + "="*50)
print(f"FEW-SHOT v2 (K={FEW_SHOT_K}, frozen encoder) — Hindi test")
print("="*50)
print(f"Accuracy  : {acc:.4f}")
print(f"Macro-F1  : {f1:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"\nBaseline  : 0.5454")
print(f"Gain      : {f1 - 0.5454:+.4f}")

del model
torch.cuda.empty_cache()

Support set: 32 | Balance: {1: 16, 0: 16}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Trainable params: 14,767,874 / 278,045,186 (5.3%)

Fine-tuning 2 epochs | LR 3e-05 | 8 steps/epoch
----------------------------------------
Epoch 1/2 — loss: 0.6734
Epoch 2/2 — loss: 0.5144

FEW-SHOT v2 (K=16, frozen encoder) — Hindi test
Accuracy  : 0.5339
Macro-F1  : 0.4657
Precision : 0.6493
Recall    : 0.5590

Baseline  : 0.5454
Gain      : -0.0797


In [29]:
# ================================================================
# FEW-SHOT v2 — frozen encoder, only top 2 layers + head updated
# ================================================================

FEW_SHOT_K      = 32
FEW_SHOT_EPOCHS = 2      # fewer epochs
FEW_SHOT_LR     = 3e-5   # higher LR is okay since fewer params are updating
FEW_SHOT_SEED   = RANDOM_SEED

# --- support set (same as before) ---
few_shot_frames = []
for label in [0, 1]:
    subset = hindi_val_df[hindi_val_df['label'] == label]
    few_shot_frames.append(subset.sample(n=min(FEW_SHOT_K, len(subset)), random_state=FEW_SHOT_SEED))

few_shot_df = pd.concat(few_shot_frames).sample(frac=1, random_state=FEW_SHOT_SEED).reset_index(drop=True)
print(f"Support set: {len(few_shot_df)} | Balance: {few_shot_df['label'].value_counts().to_dict()}")

# --- load checkpoint ---
model = AutoModelForSequenceClassification.from_pretrained(path2).to(device)
tokenizer = AutoTokenizer.from_pretrained(path2)

# ----------------------------------------------------------------
# KEY CHANGE: freeze everything except top 2 encoder layers + head
# ----------------------------------------------------------------
# First freeze all parameters
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last 2 transformer layers
for layer in model.roberta.encoder.layer[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

# Unfreeze classifier head
for param in model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

# --- fine-tune ---
fs_loader = make_loader(
    few_shot_df['text'].tolist(),
    few_shot_df['label'].tolist(),
    tokenizer, 4, MAX_SEQ_LEN, shuffle=True   # batch=4 → more gradient steps
)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=FEW_SHOT_LR, weight_decay=0.01
)
loss_fn = torch.nn.CrossEntropyLoss(weight=cw)

print(f"\nFine-tuning {FEW_SHOT_EPOCHS} epochs | LR {FEW_SHOT_LR} | {len(fs_loader)} steps/epoch")
print("-" * 40)

for epoch in range(FEW_SHOT_EPOCHS):
    model.train()
    epoch_loss = 0.0
    for batch in fs_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        loss = loss_fn(model(input_ids=input_ids, attention_mask=attention_mask).logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{FEW_SHOT_EPOCHS} — loss: {epoch_loss/len(fs_loader):.4f}")

# --- evaluate ---
model.eval()
test_loader = make_loader(
    hindi_test_df['text'].tolist(),
    hindi_test_df['label'].tolist(),
    tokenizer, BATCH_SIZE, MAX_SEQ_LEN, shuffle=False
)

_, _, _, _, test_labels, test_probs = run_eval_epoch(model, test_loader, loss_fn, device)
test_preds = (np.array(test_probs)[:, 1] >= 0.5).astype(int)

acc  = accuracy_score(test_labels, test_preds)
f1   = f1_score(test_labels, test_preds, average='macro', zero_division=0)
prec = precision_score(test_labels, test_preds, average='macro', zero_division=0)
rec  = recall_score(test_labels, test_preds, average='macro', zero_division=0)

print("\n" + "="*50)
print(f"FEW-SHOT v2 (K={FEW_SHOT_K}, frozen encoder) — Hindi test")
print("="*50)
print(f"Accuracy  : {acc:.4f}")
print(f"Macro-F1  : {f1:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"\nBaseline  : 0.5454")
print(f"Gain      : {f1 - 0.5454:+.4f}")

del model
torch.cuda.empty_cache()

Support set: 64 | Balance: {1: 32, 0: 32}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Trainable params: 14,767,874 / 278,045,186 (5.3%)

Fine-tuning 2 epochs | LR 3e-05 | 16 steps/epoch
----------------------------------------
Epoch 1/2 — loss: 0.6897
Epoch 2/2 — loss: 0.5577

FEW-SHOT v2 (K=32, frozen encoder) — Hindi test
Accuracy  : 0.5537
Macro-F1  : 0.5052
Precision : 0.6446
Recall    : 0.5760

Baseline  : 0.5454
Gain      : -0.0402


In [30]:
# ================================================================
# FEW-SHOT v2 — frozen encoder, only top 2 layers + head updated
# ================================================================

FEW_SHOT_K      = 32
FEW_SHOT_EPOCHS = 2      # fewer epochs
FEW_SHOT_LR     = 3e-5   # higher LR is okay since fewer params are updating
FEW_SHOT_SEED   = RANDOM_SEED

# --- support set (same as before) ---
few_shot_frames = []
for label in [0, 1]:
    subset = hindi_val_df[hindi_val_df['label'] == label]
    few_shot_frames.append(subset.sample(n=min(FEW_SHOT_K, len(subset)), random_state=FEW_SHOT_SEED))

few_shot_df = pd.concat(few_shot_frames).sample(frac=1, random_state=FEW_SHOT_SEED).reset_index(drop=True)
print(f"Support set: {len(few_shot_df)} | Balance: {few_shot_df['label'].value_counts().to_dict()}")

# --- load checkpoint ---
model = AutoModelForSequenceClassification.from_pretrained(path2).to(device)
tokenizer = AutoTokenizer.from_pretrained(path2)

# ----------------------------------------------------------------
# KEY CHANGE: freeze everything except top 3 encoder layers + head
# ----------------------------------------------------------------
# First freeze all parameters
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last 2 transformer layers
for layer in model.roberta.encoder.layer[-3:]:
    for param in layer.parameters():
        param.requires_grad = True

# Unfreeze classifier head
for param in model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

# --- fine-tune ---
fs_loader = make_loader(
    few_shot_df['text'].tolist(),
    few_shot_df['label'].tolist(),
    tokenizer, 4, MAX_SEQ_LEN, shuffle=True   # batch=4 → more gradient steps
)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=FEW_SHOT_LR, weight_decay=0.01
)
loss_fn = torch.nn.CrossEntropyLoss(weight=cw)

print(f"\nFine-tuning {FEW_SHOT_EPOCHS} epochs | LR {FEW_SHOT_LR} | {len(fs_loader)} steps/epoch")
print("-" * 40)

for epoch in range(FEW_SHOT_EPOCHS):
    model.train()
    epoch_loss = 0.0
    for batch in fs_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        loss = loss_fn(model(input_ids=input_ids, attention_mask=attention_mask).logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{FEW_SHOT_EPOCHS} — loss: {epoch_loss/len(fs_loader):.4f}")

# --- evaluate ---
model.eval()
test_loader = make_loader(
    hindi_test_df['text'].tolist(),
    hindi_test_df['label'].tolist(),
    tokenizer, BATCH_SIZE, MAX_SEQ_LEN, shuffle=False
)

_, _, _, _, test_labels, test_probs = run_eval_epoch(model, test_loader, loss_fn, device)
test_preds = (np.array(test_probs)[:, 1] >= 0.5).astype(int)

acc  = accuracy_score(test_labels, test_preds)
f1   = f1_score(test_labels, test_preds, average='macro', zero_division=0)
prec = precision_score(test_labels, test_preds, average='macro', zero_division=0)
rec  = recall_score(test_labels, test_preds, average='macro', zero_division=0)

print("\n" + "="*50)
print(f"FEW-SHOT v2 (K={FEW_SHOT_K}, frozen encoder) — Hindi test")
print("="*50)
print(f"Accuracy  : {acc:.4f}")
print(f"Macro-F1  : {f1:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"\nBaseline  : 0.5454")
print(f"Gain      : {f1 - 0.5454:+.4f}")

del model
torch.cuda.empty_cache()

Support set: 64 | Balance: {1: 32, 0: 32}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Trainable params: 21,855,746 / 278,045,186 (7.9%)

Fine-tuning 2 epochs | LR 3e-05 | 16 steps/epoch
----------------------------------------
Epoch 1/2 — loss: 0.8210
Epoch 2/2 — loss: 0.5387

FEW-SHOT v2 (K=32, frozen encoder) — Hindi test
Accuracy  : 0.5737
Macro-F1  : 0.5207
Precision : 0.7071
Recall    : 0.5974

Baseline  : 0.5454
Gain      : -0.0247


In [31]:
# ================================================================
# FEW-SHOT v2 — frozen encoder, only top 2 layers + head updated
# ================================================================

FEW_SHOT_K      = 64
FEW_SHOT_EPOCHS = 2      # fewer epochs
FEW_SHOT_LR     = 3e-5   # higher LR is okay since fewer params are updating
FEW_SHOT_SEED   = RANDOM_SEED

# --- support set (same as before) ---
few_shot_frames = []
for label in [0, 1]:
    subset = hindi_val_df[hindi_val_df['label'] == label]
    few_shot_frames.append(subset.sample(n=min(FEW_SHOT_K, len(subset)), random_state=FEW_SHOT_SEED))

few_shot_df = pd.concat(few_shot_frames).sample(frac=1, random_state=FEW_SHOT_SEED).reset_index(drop=True)
print(f"Support set: {len(few_shot_df)} | Balance: {few_shot_df['label'].value_counts().to_dict()}")

# --- load checkpoint ---
model = AutoModelForSequenceClassification.from_pretrained(path2).to(device)
tokenizer = AutoTokenizer.from_pretrained(path2)

# ----------------------------------------------------------------
# KEY CHANGE: freeze everything except top 3 encoder layers + head
# ----------------------------------------------------------------
# First freeze all parameters
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last 2 transformer layers
for layer in model.roberta.encoder.layer[-3:]:
    for param in layer.parameters():
        param.requires_grad = True

# Unfreeze classifier head
for param in model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

# --- fine-tune ---
fs_loader = make_loader(
    few_shot_df['text'].tolist(),
    few_shot_df['label'].tolist(),
    tokenizer, 4, MAX_SEQ_LEN, shuffle=True   # batch=4 → more gradient steps
)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=FEW_SHOT_LR, weight_decay=0.01
)
loss_fn = torch.nn.CrossEntropyLoss(weight=cw)

print(f"\nFine-tuning {FEW_SHOT_EPOCHS} epochs | LR {FEW_SHOT_LR} | {len(fs_loader)} steps/epoch")
print("-" * 40)

for epoch in range(FEW_SHOT_EPOCHS):
    model.train()
    epoch_loss = 0.0
    for batch in fs_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        loss = loss_fn(model(input_ids=input_ids, attention_mask=attention_mask).logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{FEW_SHOT_EPOCHS} — loss: {epoch_loss/len(fs_loader):.4f}")

# --- evaluate ---
model.eval()
test_loader = make_loader(
    hindi_test_df['text'].tolist(),
    hindi_test_df['label'].tolist(),
    tokenizer, BATCH_SIZE, MAX_SEQ_LEN, shuffle=False
)

_, _, _, _, test_labels, test_probs = run_eval_epoch(model, test_loader, loss_fn, device)
test_preds = (np.array(test_probs)[:, 1] >= 0.5).astype(int)

acc  = accuracy_score(test_labels, test_preds)
f1   = f1_score(test_labels, test_preds, average='macro', zero_division=0)
prec = precision_score(test_labels, test_preds, average='macro', zero_division=0)
rec  = recall_score(test_labels, test_preds, average='macro', zero_division=0)

print("\n" + "="*50)
print(f"FEW-SHOT v2 (K={FEW_SHOT_K}, frozen encoder) — Hindi test")
print("="*50)
print(f"Accuracy  : {acc:.4f}")
print(f"Macro-F1  : {f1:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"\nBaseline  : 0.5454")
print(f"Gain      : {f1 - 0.5454:+.4f}")

del model
torch.cuda.empty_cache()

Support set: 128 | Balance: {0: 64, 1: 64}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Trainable params: 21,855,746 / 278,045,186 (7.9%)

Fine-tuning 2 epochs | LR 3e-05 | 32 steps/epoch
----------------------------------------
Epoch 1/2 — loss: 0.6739
Epoch 2/2 — loss: 0.5541

FEW-SHOT v2 (K=64, frozen encoder) — Hindi test
Accuracy  : 0.6703
Macro-F1  : 0.6527
Precision : 0.7535
Recall    : 0.6874

Baseline  : 0.5454
Gain      : +0.1073


In [32]:
# ================================================================
# FEW-SHOT v2 — frozen encoder, only top 2 layers + head updated
# ================================================================

FEW_SHOT_K      = 128
FEW_SHOT_EPOCHS = 2      # fewer epochs
FEW_SHOT_LR     = 3e-5   # higher LR is okay since fewer params are updating
FEW_SHOT_SEED   = RANDOM_SEED

# --- support set (same as before) ---
few_shot_frames = []
for label in [0, 1]:
    subset = hindi_val_df[hindi_val_df['label'] == label]
    few_shot_frames.append(subset.sample(n=min(FEW_SHOT_K, len(subset)), random_state=FEW_SHOT_SEED))

few_shot_df = pd.concat(few_shot_frames).sample(frac=1, random_state=FEW_SHOT_SEED).reset_index(drop=True)
print(f"Support set: {len(few_shot_df)} | Balance: {few_shot_df['label'].value_counts().to_dict()}")

# --- load checkpoint ---
model = AutoModelForSequenceClassification.from_pretrained(path2).to(device)
tokenizer = AutoTokenizer.from_pretrained(path2)

# ----------------------------------------------------------------
# KEY CHANGE: freeze everything except top 3 encoder layers + head
# ----------------------------------------------------------------
# First freeze all parameters
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last 2 transformer layers
for layer in model.roberta.encoder.layer[-3:]:
    for param in layer.parameters():
        param.requires_grad = True

# Unfreeze classifier head
for param in model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

# --- fine-tune ---
fs_loader = make_loader(
    few_shot_df['text'].tolist(),
    few_shot_df['label'].tolist(),
    tokenizer, 4, MAX_SEQ_LEN, shuffle=True   # batch=4 → more gradient steps
)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=FEW_SHOT_LR, weight_decay=0.01
)
loss_fn = torch.nn.CrossEntropyLoss(weight=cw)

print(f"\nFine-tuning {FEW_SHOT_EPOCHS} epochs | LR {FEW_SHOT_LR} | {len(fs_loader)} steps/epoch")
print("-" * 40)

for epoch in range(FEW_SHOT_EPOCHS):
    model.train()
    epoch_loss = 0.0
    for batch in fs_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        loss = loss_fn(model(input_ids=input_ids, attention_mask=attention_mask).logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{FEW_SHOT_EPOCHS} — loss: {epoch_loss/len(fs_loader):.4f}")

# --- evaluate ---
model.eval()
test_loader = make_loader(
    hindi_test_df['text'].tolist(),
    hindi_test_df['label'].tolist(),
    tokenizer, BATCH_SIZE, MAX_SEQ_LEN, shuffle=False
)

_, _, _, _, test_labels, test_probs = run_eval_epoch(model, test_loader, loss_fn, device)
test_preds = (np.array(test_probs)[:, 1] >= 0.5).astype(int)

acc  = accuracy_score(test_labels, test_preds)
f1   = f1_score(test_labels, test_preds, average='macro', zero_division=0)
prec = precision_score(test_labels, test_preds, average='macro', zero_division=0)
rec  = recall_score(test_labels, test_preds, average='macro', zero_division=0)

print("\n" + "="*50)
print(f"FEW-SHOT v2 (K={FEW_SHOT_K}, frozen encoder) — Hindi test")
print("="*50)
print(f"Accuracy  : {acc:.4f}")
print(f"Macro-F1  : {f1:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"\nBaseline  : 0.5454")
print(f"Gain      : {f1 - 0.5454:+.4f}")

del model
torch.cuda.empty_cache()

Support set: 204 | Balance: {0: 109, 1: 95}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Trainable params: 21,855,746 / 278,045,186 (7.9%)

Fine-tuning 2 epochs | LR 3e-05 | 51 steps/epoch
----------------------------------------
Epoch 1/2 — loss: 0.6283
Epoch 2/2 — loss: 0.4759

FEW-SHOT v2 (K=128, frozen encoder) — Hindi test
Accuracy  : 0.7823
Macro-F1  : 0.7814
Precision : 0.7814
Recall    : 0.7815

Baseline  : 0.5454
Gain      : +0.2360


In [33]:
# ================================================================
# PSEUDO-LABELING — HingRoBERTa self-training on Hindi
# ================================================================
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
import numpy as np
import torch
import pandas as pd

# ----------------------------------------------------------------
# Config
# ----------------------------------------------------------------
PSEUDO_CONFIDENCE_THRESHOLD = 0.85   # only keep predictions this confident
PSEUDO_LR                   = 2e-5   # back to full training LR (more data now)
PSEUDO_EPOCHS               = 3
PSEUDO_BATCH_SIZE           = 16

# ----------------------------------------------------------------
# Step 1: Load HingRoBERTa checkpoint, run inference on Hindi test
#         NOTE: we use hindi_test_df here intentionally — we are
#         not evaluating yet, we are generating pseudo labels.
#         Final eval still uses the same hindi_test_df AFTER retraining.
# ----------------------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(path2).to(device)
tokenizer = AutoTokenizer.from_pretrained(path2)
model.eval()

pseudo_loader = make_loader(
    hindi_test_df['text'].tolist(),
    hindi_test_df['label'].tolist(),   # labels ignored during inference
    tokenizer, BATCH_SIZE, MAX_SEQ_LEN, shuffle=False
)

loss_fn = torch.nn.CrossEntropyLoss(weight=cw)

_, _, _, _, _, all_probs = run_eval_epoch(model, pseudo_loader, loss_fn, device)
all_probs = np.array(all_probs)   # shape: (N, 2)

del model
torch.cuda.empty_cache()

# ----------------------------------------------------------------
# Step 2: Filter by confidence — keep only high-confidence preds
# ----------------------------------------------------------------
max_probs   = all_probs.max(axis=1)          # confidence = max class prob
pseudo_preds = all_probs.argmax(axis=1)      # predicted label

confident_mask = max_probs >= PSEUDO_CONFIDENCE_THRESHOLD

pseudo_texts  = hindi_test_df['text'].values[confident_mask]
pseudo_labels = pseudo_preds[confident_mask]
pseudo_confs  = max_probs[confident_mask]

print(f"Total Hindi test rows     : {len(hindi_test_df)}")
print(f"Confident predictions     : {confident_mask.sum()} ({100*confident_mask.mean():.1f}%)")
print(f"Pseudo label balance      : 0={( pseudo_labels==0).sum()}  1={(pseudo_labels==1).sum()}")
print(f"Mean confidence (kept)    : {pseudo_confs.mean():.4f}")
print(f"Min  confidence (kept)    : {pseudo_confs.min():.4f}")

# ----------------------------------------------------------------
# Step 3: Build augmented training set
#         original Hinglish train + pseudo-labeled Hindi
# ----------------------------------------------------------------

# train_df is your original normalized Hinglish training data
pseudo_df = pd.DataFrame({'text': pseudo_texts, 'label': pseudo_labels})

augmented_df = pd.concat([train_df, pseudo_df], ignore_index=True).sample(
    frac=1, random_state=RANDOM_SEED
).reset_index(drop=True)

print(f"\nOriginal train size  : {len(train_df)}")
print(f"Pseudo-labeled added : {len(pseudo_df)}")
print(f"Augmented train size : {len(augmented_df)}")
print(f"Augmented balance    : {augmented_df['label'].value_counts().to_dict()}")

# ----------------------------------------------------------------
# Step 4: Retrain HingRoBERTa from scratch on augmented data
#         (reload checkpoint — don't use the inference model)
# ----------------------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(path2).to(device)
tokenizer = AutoTokenizer.from_pretrained(path2)

# split augmented data into train/val (90/10)
from sklearn.model_selection import train_test_split
aug_train_df, aug_val_df = train_test_split(
    augmented_df, test_size=0.10,
    random_state=RANDOM_SEED,
    stratify=augmented_df['label']
)

aug_train_loader = make_loader(
    aug_train_df['text'].tolist(), aug_train_df['label'].tolist(),
    tokenizer, PSEUDO_BATCH_SIZE, MAX_SEQ_LEN, shuffle=True
)
aug_val_loader = make_loader(
    aug_val_df['text'].tolist(), aug_val_df['label'].tolist(),
    tokenizer, PSEUDO_BATCH_SIZE, MAX_SEQ_LEN, shuffle=False
)

optimizer = AdamW(model.parameters(), lr=PSEUDO_LR, weight_decay=0.01)
total_steps = len(aug_train_loader) * PSEUDO_EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# use unweighted loss for retraining — augmented set is more balanced
retrain_loss_fn = torch.nn.CrossEntropyLoss()

print(f"\nRetraining on augmented set...")
print(f"Steps/epoch: {len(aug_train_loader)} | Total steps: {total_steps}")
print("-" * 40)

best_val_f1 = 0.0
best_state  = None

for epoch in range(PSEUDO_EPOCHS):
    # --- train ---
    model.train()
    train_loss = 0.0
    for batch in aug_train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        loss = retrain_loss_fn(
            model(input_ids=input_ids, attention_mask=attention_mask).logits,
            labels
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()

    # --- val ---
    model.eval()
    _, _, _, _, val_labels_epoch, val_probs_epoch = run_eval_epoch(
        model, aug_val_loader, retrain_loss_fn, device
    )
    val_preds_epoch = np.array(val_probs_epoch).argmax(axis=1)
    val_f1 = f1_score(val_labels_epoch, val_preds_epoch, average='macro', zero_division=0)

    print(f"Epoch {epoch+1}/{PSEUDO_EPOCHS} — train loss: {train_loss/len(aug_train_loader):.4f} | val macro-F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

print(f"\nBest val macro-F1 during retraining: {best_val_f1:.4f}")

# ----------------------------------------------------------------
# Step 5: Evaluate best checkpoint on hindi_test_df
# ----------------------------------------------------------------
model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
model.eval()

test_loader = make_loader(
    hindi_test_df['text'].tolist(),
    hindi_test_df['label'].tolist(),
    tokenizer, BATCH_SIZE, MAX_SEQ_LEN, shuffle=False
)

_, _, _, _, test_labels, test_probs = run_eval_epoch(model, test_loader, retrain_loss_fn, device)
test_preds = (np.array(test_probs)[:, 1] >= 0.5).astype(int)

acc  = accuracy_score(test_labels, test_preds)
f1   = f1_score(test_labels, test_preds, average='macro', zero_division=0)
prec = precision_score(test_labels, test_preds, average='macro', zero_division=0)
rec  = recall_score(test_labels, test_preds, average='macro', zero_division=0)

print("\n" + "="*50)
print(f"PSEUDO-LABELING (conf≥{PSEUDO_CONFIDENCE_THRESHOLD}) — Hindi test")
print("="*50)
print(f"Accuracy  : {acc:.4f}")
print(f"Macro-F1  : {f1:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"\nBaseline (normalized)     : 0.5454")
print(f"+ Threshold optimization  : 0.5454")
print(f"+ Pseudo-labeling         : {f1:.4f}  ({f1-0.5454:+.4f})")

del model
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Total Hindi test rows     : 7988
Confident predictions     : 511 (6.4%)
Pseudo label balance      : 0=371  1=140
Mean confidence (kept)    : 0.9074
Min  confidence (kept)    : 0.8502

Original train size  : 33984
Pseudo-labeled added : 511
Augmented train size : 34495
Augmented balance    : {0: 24648, 1: 9847}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Retraining on augmented set...
Steps/epoch: 1941 | Total steps: 5823
----------------------------------------
Epoch 1/3 — train loss: 0.1967 | val macro-F1: 0.9247
Epoch 2/3 — train loss: 0.1560 | val macro-F1: 0.9282
Epoch 3/3 — train loss: 0.1137 | val macro-F1: 0.9267

Best val macro-F1 during retraining: 0.9282

PSEUDO-LABELING (conf≥0.85) — Hindi test
Accuracy  : 0.5260
Macro-F1  : 0.4997
Precision : 0.5159
Recall    : 0.5135

Baseline (normalized)     : 0.5454
+ Threshold optimization  : 0.5454
+ Pseudo-labeling         : 0.4997  (-0.0457)


In [34]:
# ================================================================
# PSEUDO-LABELING — HingRoBERTa self-training on Hindi
# ================================================================
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
import numpy as np
import torch
import pandas as pd

# ----------------------------------------------------------------
# Config
# ----------------------------------------------------------------
PSEUDO_CONFIDENCE_THRESHOLD = 0.85   # only keep predictions this confident
PSEUDO_LR                   = 2e-5   # back to full training LR (more data now)
PSEUDO_EPOCHS               = 3
PSEUDO_BATCH_SIZE           = 16

# ----------------------------------------------------------------
# Step 1: Load HingRoBERTa checkpoint, run inference on Hindi test
#         NOTE: we use hindi_test_df here intentionally — we are
#         not evaluating yet, we are generating pseudo labels.
#         Final eval still uses the same hindi_test_df AFTER retraining.
# ----------------------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(path2).to(device)
tokenizer = AutoTokenizer.from_pretrained(path2)
model.eval()

pseudo_loader = make_loader(
    hindi_test_df['text'].tolist(),
    hindi_test_df['label'].tolist(),   # labels ignored during inference
    tokenizer, BATCH_SIZE, MAX_SEQ_LEN, shuffle=False
)

loss_fn = torch.nn.CrossEntropyLoss(weight=cw)

_, _, _, _, _, all_probs = run_eval_epoch(model, pseudo_loader, loss_fn, device)
all_probs = np.array(all_probs)   # shape: (N, 2)

del model
torch.cuda.empty_cache()

# ----------------------------------------------------------------
# Step 2: Filter by confidence — keep only high-confidence preds
# ----------------------------------------------------------------
# ----------------------------------------------------------------
# Step 2 REVISED: per-class confidence filtering
# ----------------------------------------------------------------
PSEUDO_CONFIDENCE_THRESHOLD = 0.70
MAX_PSEUDO_PER_CLASS = 2000   # cap to avoid drowning out hate class

all_probs    = np.array(all_probs)
max_probs    = all_probs.max(axis=1)
pseudo_preds = all_probs.argmax(axis=1)

confident_mask = max_probs >= PSEUDO_CONFIDENCE_THRESHOLD

pseudo_texts_all  = hindi_test_df['text'].values[confident_mask]
pseudo_labels_all = pseudo_preds[confident_mask]
pseudo_confs_all  = max_probs[confident_mask]

# cap each class separately
kept_indices = []
for label in [0, 1]:
    idx = np.where(pseudo_labels_all == label)[0]
    # sort by confidence descending, take top N
    idx_sorted = idx[np.argsort(-pseudo_confs_all[idx])]
    kept_indices.extend(idx_sorted[:MAX_PSEUDO_PER_CLASS].tolist())

kept_indices  = sorted(kept_indices)
pseudo_texts  = pseudo_texts_all[kept_indices]
pseudo_labels = pseudo_labels_all[kept_indices]
pseudo_confs  = pseudo_confs_all[kept_indices]

print(f"Total Hindi test rows     : {len(hindi_test_df)}")
print(f"After confidence filter   : {confident_mask.sum()} ({100*confident_mask.mean():.1f}%)")
print(f"After per-class cap       : {len(pseudo_texts)}")
print(f"Pseudo label balance      : 0={(pseudo_labels==0).sum()}  1={(pseudo_labels==1).sum()}")
print(f"Mean confidence (kept)    : {pseudo_confs.mean():.4f}")

# ----------------------------------------------------------------
# Step 3: Build augmented training set
#         original Hinglish train + pseudo-labeled Hindi
# ----------------------------------------------------------------

# train_df is your original normalized Hinglish training data
pseudo_df = pd.DataFrame({'text': pseudo_texts, 'label': pseudo_labels})

augmented_df = pd.concat([train_df, pseudo_df], ignore_index=True).sample(
    frac=1, random_state=RANDOM_SEED
).reset_index(drop=True)

print(f"\nOriginal train size  : {len(train_df)}")
print(f"Pseudo-labeled added : {len(pseudo_df)}")
print(f"Augmented train size : {len(augmented_df)}")
print(f"Augmented balance    : {augmented_df['label'].value_counts().to_dict()}")

# ----------------------------------------------------------------
# Step 4: Retrain HingRoBERTa from scratch on augmented data
#         (reload checkpoint — don't use the inference model)
# ----------------------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(path2).to(device)
tokenizer = AutoTokenizer.from_pretrained(path2)

# split augmented data into train/val (90/10)
from sklearn.model_selection import train_test_split
aug_train_df, aug_val_df = train_test_split(
    augmented_df, test_size=0.10,
    random_state=RANDOM_SEED,
    stratify=augmented_df['label']
)

aug_train_loader = make_loader(
    aug_train_df['text'].tolist(), aug_train_df['label'].tolist(),
    tokenizer, PSEUDO_BATCH_SIZE, MAX_SEQ_LEN, shuffle=True
)
aug_val_loader = make_loader(
    aug_val_df['text'].tolist(), aug_val_df['label'].tolist(),
    tokenizer, PSEUDO_BATCH_SIZE, MAX_SEQ_LEN, shuffle=False
)

optimizer = AdamW(model.parameters(), lr=PSEUDO_LR, weight_decay=0.01)
total_steps = len(aug_train_loader) * PSEUDO_EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# use unweighted loss for retraining — augmented set is more balanced
retrain_loss_fn = torch.nn.CrossEntropyLoss()

print(f"\nRetraining on augmented set...")
print(f"Steps/epoch: {len(aug_train_loader)} | Total steps: {total_steps}")
print("-" * 40)

best_val_f1 = 0.0
best_state  = None

for epoch in range(PSEUDO_EPOCHS):
    # --- train ---
    model.train()
    train_loss = 0.0
    for batch in aug_train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        loss = retrain_loss_fn(
            model(input_ids=input_ids, attention_mask=attention_mask).logits,
            labels
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()

    # --- val ---
    model.eval()
    _, _, _, _, val_labels_epoch, val_probs_epoch = run_eval_epoch(
        model, aug_val_loader, retrain_loss_fn, device
    )
    val_preds_epoch = np.array(val_probs_epoch).argmax(axis=1)
    val_f1 = f1_score(val_labels_epoch, val_preds_epoch, average='macro', zero_division=0)

    print(f"Epoch {epoch+1}/{PSEUDO_EPOCHS} — train loss: {train_loss/len(aug_train_loader):.4f} | val macro-F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

print(f"\nBest val macro-F1 during retraining: {best_val_f1:.4f}")

# ----------------------------------------------------------------
# Step 5: Evaluate best checkpoint on hindi_test_df
# ----------------------------------------------------------------
model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
model.eval()

test_loader = make_loader(
    hindi_test_df['text'].tolist(),
    hindi_test_df['label'].tolist(),
    tokenizer, BATCH_SIZE, MAX_SEQ_LEN, shuffle=False
)

_, _, _, _, test_labels, test_probs = run_eval_epoch(model, test_loader, retrain_loss_fn, device)
test_preds = (np.array(test_probs)[:, 1] >= 0.5).astype(int)

acc  = accuracy_score(test_labels, test_preds)
f1   = f1_score(test_labels, test_preds, average='macro', zero_division=0)
prec = precision_score(test_labels, test_preds, average='macro', zero_division=0)
rec  = recall_score(test_labels, test_preds, average='macro', zero_division=0)

print("\n" + "="*50)
print(f"PSEUDO-LABELING (conf≥{PSEUDO_CONFIDENCE_THRESHOLD}) — Hindi test")
print("="*50)
print(f"Accuracy  : {acc:.4f}")
print(f"Macro-F1  : {f1:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"\nBaseline (normalized)     : 0.5454")
print(f"+ Threshold optimization  : 0.5454")
print(f"+ Pseudo-labeling         : {f1:.4f}  ({f1-0.5454:+.4f})")

del model
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Total Hindi test rows     : 7988
After confidence filter   : 2390 (29.9%)
After per-class cap       : 2390
Pseudo label balance      : 0=1562  1=828
Mean confidence (kept)    : 0.7915

Original train size  : 33984
Pseudo-labeled added : 2390
Augmented train size : 36374
Augmented balance    : {0: 25839, 1: 10535}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Retraining on augmented set...
Steps/epoch: 2046 | Total steps: 6138
----------------------------------------
Epoch 1/3 — train loss: 0.2004 | val macro-F1: 0.9166
Epoch 2/3 — train loss: 0.1566 | val macro-F1: 0.9230
Epoch 3/3 — train loss: 0.1141 | val macro-F1: 0.9265

Best val macro-F1 during retraining: 0.9265

PSEUDO-LABELING (conf≥0.7) — Hindi test
Accuracy  : 0.5572
Macro-F1  : 0.5547
Precision : 0.5549
Recall    : 0.5547

Baseline (normalized)     : 0.5454
+ Threshold optimization  : 0.5454
+ Pseudo-labeling         : 0.5547  (+0.0093)


---
## XAI Module Reference Card

### Directory layout after running this notebook
```
roberta_outputs/
├── full_results.csv
├── HingRoBERTa/
│   ├── best_checkpoint/          ← reload with AutoModelForSequenceClassification
│   ├── training_curves.png
│   ├── cm_english.png
│   ├── cm_hindi.png
│   ├── predictions_english.csv
│   ├── predictions_hindi.csv
│   ├── xai_english/
│   │   ├── xai_samples.npz       ← main XAI arrays
│   │   ├── tokens_list.npy
│   │   ├── token_rankings.npy    ← AOPC input (attention-ranked)
│   │   ├── last_layer_attn.npy
│   │   ├── all_layers_attn.npy
│   │   ├── aopc_attention.npz
│   │   └── aopc_attention.png
│   └── xai_hindi/  (same structure)
└── XLM_RoBERTa/ (same structure)
```

### Loading in `xai/explain.py`
```python
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_PATH = 'roberta_outputs/HingRoBERTa/best_checkpoint'
XAI_PATH   = 'roberta_outputs/HingRoBERTa/xai_english'

model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, output_attentions=True, output_hidden_states=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Fixed-shape arrays
xai           = np.load(f'{XAI_PATH}/xai_samples.npz', allow_pickle=True)
texts         = xai['texts']           # (N,) raw strings
true_labels   = xai['true_labels']     # (N,)
pred_labels   = xai['pred_labels']     # (N,)
probs         = xai['probs']           # (N, 2)
cls_attn_rows = xai['cls_attn_rows']   # (N, seq_len) → attention bars
cls_hiddens   = xai['cls_hiddens']     # (N, 768)     → SHAP feature space

# Variable-length arrays
tokens_list     = np.load(f'{XAI_PATH}/tokens_list.npy',    allow_pickle=True)  # (N,) lists
token_rankings  = np.load(f'{XAI_PATH}/token_rankings.npy', allow_pickle=True)  # (N,) ranked idx
last_attn       = np.load(f'{XAI_PATH}/last_layer_attn.npy',allow_pickle=True)  # (N,) (heads,seq,seq)
all_attn        = np.load(f'{XAI_PATH}/all_layers_attn.npy',allow_pickle=True)  # (N,) (layers,seq,seq)
```

### SHAP text explainer
```python
import shap, torch

def predict_fn(text_list):
    enc = tokenizer(list(text_list), return_tensors='pt',
                    padding=True, truncation=True, max_length=128)
    with torch.no_grad():
        out = model(**enc)
    return torch.softmax(out.logits, -1).numpy()

explainer   = shap.Explainer(predict_fn, shap.maskers.Text(tokenizer))
shap_values = explainer(texts[:50])

# Extract SHAP token rankings for SHAP-AOPC
shap_rankings = [
    np.argsort(np.abs(shap_values[i, :, 1].values))[::-1].tolist()
    for i in range(len(texts[:50]))
]
# Then pass to compute_aopc(..., shap_rankings=shap_rankings, label='shap')
```

### Key flag reminder
Both models are saved with `output_attentions=True` and `output_hidden_states=True` in their config.
These flags are also passed explicitly at inference time in this notebook.
Your XAI teammate does **not** need to change anything — just reload and call `model(**enc, output_attentions=True)`.